# Global predictability ceilings for lag-based emulation of NASA POWER soil surface wetness

Global panel (~250 POWER cells). The linear baseline vs. the ceiling is the paper's spine.

**Inputs:** six GEE exports `global_power_cells_0.csv` … `global_power_cells_5.csv`

**Every heavy cell checkpoints per grid cell.** If Colab recycles, restore the zip and rerun the
same cell — completed cells are skipped.

Run order: 0 → 1 → 2 → **Z** → 3 → 4 → 5 → **Z** → 6 → 7 → 7b → **Z** → 8 → 9 → 10
(**Z** = the checkpoint cell; run it often.)


## Cell 0 — setup

In [ ]:
USE_DRIVE = False      # True to persist across runtime restarts

from pathlib import Path
import sys
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    assert Path("/content/drive/MyDrive").exists(), "Drive did not mount"
    %cd /content/drive/MyDrive
    !mkdir -p ssw_global/data
    %cd ssw_global

Path("data").mkdir(exist_ok=True)
NEED = [f"gpc_{i}.csv" for i in range(24)]
missing = [f for f in NEED if not (Path("data")/f).exists()]

if missing and IN_COLAB:
    hits = list(Path("/content/drive/MyDrive").rglob("gpc_*.csv")) \
           if Path("/content/drive/MyDrive").exists() else []
    for h in hits:
        if h.name in missing: (Path("data")/h.name).write_bytes(h.read_bytes())
    missing = [f for f in NEED if not (Path("data")/f).exists()]
    if missing:
        print(f"Upload the {len(missing)} chunk CSVs:", missing[:5], "...")
        from google.colab import files
        up = files.upload()
        for k in up: (Path("data")/k).write_bytes(up[k])

!pip install -q pyarrow lightgbm xgboost statsmodels

print("\n--- SETUP ---")
ok = True
for f in NEED:
    p = Path("data")/f
    ok &= p.exists()
print(f"  {sum((Path('data')/f).exists() for f in NEED)}/24 chunk CSVs present")
assert ok, "Missing chunk CSVs — rerun and upload all 24"
print("ready")

Upload the 24 chunk CSVs: ['gpc_0.csv', 'gpc_1.csv', 'gpc_2.csv', 'gpc_3.csv', 'gpc_4.csv'] ...


Saving drive-download-20260822T163700Z-1-001.zip to drive-download-20260822T163700Z-1-001.zip

--- SETUP ---
  0/24 chunk CSVs present


AssertionError: Missing chunk CSVs — rerun and upload all 24

In [ ]:

import shutil
from pathlib import Path
OUT = Path("data")

z = next(OUT.glob("*.zip"), None) or next(Path(".").glob("drive-download*.zip"), None)
assert z, "zip not found"
shutil.unpack_archive(str(z), OUT)

# GEE sometimes nests them in a folder — flatten if so
for p in list(OUT.rglob("gpc_*.csv")):
    if p.parent != OUT: p.rename(OUT/p.name)

n = len(list(OUT.glob("gpc_*.csv")))
print(f"{n}/24 chunk CSVs present")
assert n == 24, "unpack incomplete"
print("ready")

24/24 chunk CSVs present
ready


## Cell Z — checkpoint (run often)

Zips `data/` and downloads it. Restore with the cell below it.

In [ ]:
import shutil
from pathlib import Path
shutil.make_archive("ssw_global_results", "zip", "data")
try:
    from google.colab import files; files.download("ssw_global_results.zip")
    print("downloaded ssw_global_results.zip")
except ImportError:
    print("saved", Path("ssw_global_results.zip").resolve())

# --- TO RESTORE after a recycle, run this instead: ---
# from google.colab import files; import shutil
# from pathlib import Path; Path("data").mkdir(exist_ok=True)
# up = files.upload(); shutil.unpack_archive("ssw_global_results.zip", "data")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloaded ssw_global_results.zip


## Cell 1 — global panel

Tested against the real 24-chunk export (50,860 rows). Handles three things the raw export
needs: chunk-boundary duplicates, object dtype from near-empty chunks, and PET_ann = 0 in cold
cells (which sends the aridity index to ~15,000).

Replication cells (Punjab) are forced in first and exempt from the separation rule — Moga and
Firozpur are 59.6 km apart, just inside one grid step.

**Verified on your data:** 250 cells, 29 strata, corr(AI, irrig) = -0.075, min separation 55.1 km.


In [ ]:
import numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
OUT = Path("data")
TARGET, MIN_PER_STRATUM, T_MIN, MIN_SEP_KM, R = 250, 4, 5.0, 55.0, 6371.0

g = pd.concat([pd.read_csv(f) for f in sorted(OUT.glob("gpc_*.csv"))], ignore_index=True)
n0 = len(g)
NUM = ["lat","lon","land_frac","crop_frac","AI","P_ann","PET_ann","T2M_ann","ndvi_dry_crop"]
for c in NUM: g[c] = pd.to_numeric(g[c], errors="coerce")   # near-empty chunks -> object dtype
g = g.dropna(subset=["lat","lon","AI","T2M_ann","PET_ann"])
g = g.drop_duplicates(subset=["lat","lon"])                  # chunk boundaries overlap
n1 = len(g)
# PET_ann == 0 in cold cells sends AI to ~15000; T and PET filters both required
g = g[(g.land_frac>0.2) & (g.T2M_ann>=T_MIN) & (g.PET_ann>200) & (g.P_ann>0)].reset_index(drop=True)

g["crop_frac"]   = g.crop_frac.fillna(0.0)
g["irrig_index"] = np.where(g.crop_frac<0.02, 0.0, g.ndvi_dry_crop.fillna(0.0)*g.crop_frac)
cropped = g.irrig_index > 0
q1, q2 = g.loc[cropped,"irrig_index"].quantile([1/3, 2/3])
g["irrig_class"] = np.select([~cropped, g.irrig_index<=q1, g.irrig_index<=q2],
                             ["low","low","med"], default="high")
g["aridity_class"] = pd.cut(g.AI, [-np.inf,0.05,0.20,0.50,0.65,np.inf],
                            labels=["hyperarid","arid","semiarid","dry-subhumid","humid"])
g["hemi"] = np.where(g.lat>=0, "N", "S")
g["stratum"] = g.aridity_class.astype(str) + "|" + g.irrig_class + "|" + g.hemi

PUNJAB = {"Moga":(30.816,75.171),"Hoshiarpur":(31.514,75.911),"Firozpur":(30.923,74.610)}
g["name"], g["replication"] = None, False
for nm,(la,lo) in PUNJAB.items():
    i = np.hypot(g.lat-la, g.lon-lo).idxmin(); g.loc[i,"name"]=nm; g.loc[i,"replication"]=True

def xyz(df):
    la, lo = np.radians(df.lat.to_numpy()), np.radians(df.lon.to_numpy())
    return np.column_stack([np.cos(la)*np.cos(lo), np.cos(la)*np.sin(lo), np.sin(la)])

def fp(df, k, seed):
    P = xyz(df); ch = list(seed) or [0]
    while len(ch) < min(k, len(df)):
        d = np.min(np.linalg.norm(P[:,None,:]-P[ch][None,:,:], axis=2), axis=1); d[ch] = -1
        ch.append(int(np.argmax(d)))
    return ch

counts = g.stratum.value_counts()
alloc = {s: min(MIN_PER_STRATUM, n) for s,n in counts.items()}
left = TARGET - sum(alloc.values()); CAP = max(MIN_PER_STRATUM+2, TARGET//len(counts)+2)
while left > 0:
    room = [s for s in counts.index if alloc[s] < min(counts[s], CAP)]
    if not room: CAP += 2; continue
    for s in room:
        if left <= 0: break
        alloc[s] += 1; left -= 1

kept  = list(g.index[g.replication])          # replication cells forced in, exempt from spacing
keptP = list(xyz(g.loc[kept])) if kept else []
def far(i):
    if not keptP: return True
    return (np.linalg.norm(np.array(keptP)-xyz(g.loc[[i]])[0], axis=1)*R).min() >= MIN_SEP_KM

for s_, k in alloc.items():
    sub = g[g.stratum==s_]
    if len(sub) == 0: continue
    have = sum(1 for i in kept if g.at[i,"stratum"]==s_)
    order = fp(sub.reset_index(), min(len(sub), k*4), [])
    for i in sub.reset_index().loc[order,"index"].tolist():
        if have >= k: break
        if i in kept: continue
        if far(i): kept.append(i); keptP.append(xyz(g.loc[[i]])[0]); have += 1

panel = g.loc[sorted(set(kept))].copy().reset_index(drop=True)
panel["cell_id"] = [f"{a:+07.3f}_{b:+08.3f}".replace(".","p") for a,b in zip(panel.lat, panel.lon)]
panel.to_parquet(OUT/"panel_global.parquet", index=False)

print("--- SANITY ---")
print(f"raw {n0} -> dedup {n1} -> land+warm+validPET {len(g)} -> panel {len(panel)} (target {TARGET})")
print("dupes:", panel.duplicated(["lat","lon"]).sum(), "| Punjab:", int(panel.replication.sum()), "(want 3)")
print("strata: %d of %d" % (panel.stratum.nunique(), g.stratum.nunique()))
print("\naridity x irrigation:\n", pd.crosstab(panel.aridity_class, panel.irrig_class).to_string())
print("\nhemisphere:", panel.hemi.value_counts().to_dict())
print("AI %.3f-%.3f | P %.0f-%.0f mm | PET %.0f-%.0f | T %.1f-%.1f C"
      % (panel.AI.min(),panel.AI.max(),panel.P_ann.min(),panel.P_ann.max(),
         panel.PET_ann.min(),panel.PET_ann.max(),panel.T2M_ann.min(),panel.T2M_ann.max()))
r_,p_ = stats.pearsonr(panel.AI, panel.irrig_index)
rl,pl = stats.pearsonr(np.log(panel.AI), panel.irrig_index)
print("\n*** corr(AI, irrig_index)    = %.3f (p=%.2g)" % (r_,p_))
print("*** corr(logAI, irrig_index) = %.3f (p=%.2g)   <- want |r| < 0.6" % (rl,pl))
Pm = xyz(panel); D = np.linalg.norm(Pm[:,None,:]-Pm[None,:,:], axis=2); np.fill_diagonal(D, np.inf)
print("min separation: %.1f km" % (D.min()*R))
print("\nPunjab:\n", panel[panel.replication][["name","lat","lon","AI","aridity_class","irrig_class"]].round(3).to_string(index=False))

/tmp/ipykernel_2238/3873713603.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  g = pd.concat([pd.read_csv(f) for f in sorted(OUT.glob("gpc_*.csv"))], ignore_index=True)


--- SANITY ---
raw 50860 -> dedup 50412 -> land+warm+validPET 29951 -> panel 250 (target 250)
dupes: 0 | Punjab: 3 (want 3)
strata: 29 of 29

aridity x irrigation:
 irrig_class    high  low  med
aridity_class                
hyperarid         8   18   10
arid             17   18   17
semiarid         18   18   18
dry-subhumid     18   18   18
humid            18   18   18

hemisphere: {'N': 134, 'S': 116}
AI 0.001-2.828 | P 1-4021 mm | PET 411-2523 | T 5.1-29.8 C

*** corr(AI, irrig_index)    = -0.075 (p=0.23)
*** corr(logAI, irrig_index) = 0.006 (p=0.92)   <- want |r| < 0.6
min separation: 55.1 km

Punjab:
       name  lat    lon    AI aridity_class irrig_class
  Firozpur 31.0 74.688 0.353      semiarid        high
      Moga 31.0 75.312 0.411      semiarid        high
Hoshiarpur 31.5 75.938 0.656         humid        high


## Cell 2 — POWER daily pull

~250 cells, checkpointed to `data/raw_daily/`. Budget 10–20 min. **Run Cell Z after this.**

In [ ]:
import requests, time
RAW = OUT/"raw_daily"; RAW.mkdir(parents=True, exist_ok=True)
panel = pd.read_parquet(OUT/"panel_global.parquet").reset_index(drop=True)

PARAMS = ["GWETTOP","GWETROOT","GWETPROF","PRECTOTCORR","T2M","T2M_MAX","T2M_MIN",
          "RH2M","WS2M","ALLSKY_SFC_SW_DWN","EVLAND","PS"]
URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
START, END = "19810101", "20251231"

done, failed = 0, []
for i, r in panel.iterrows():
    fp_ = RAW/f"{r.cell_id}.parquet"
    if fp_.exists(): done += 1; continue
    try:
        q = dict(parameters=",".join(PARAMS), community="AG", latitude=float(r.lat),
                 longitude=float(r.lon), start=START, end=END, format="JSON")
        resp = requests.get(URL, params=q, timeout=300); resp.raise_for_status()
        df = pd.DataFrame(resp.json()["properties"]["parameter"])
        df.index = pd.to_datetime(df.index, format="%Y%m%d")
        df = df.replace(-999.0, np.nan).rename_axis("date").reset_index()
        df.insert(0, "cell_id", r.cell_id)
        df.insert(1, "name", r["name"] if pd.notna(r["name"]) else r.cell_id)
        df.to_parquet(fp_, index=False); done += 1; time.sleep(1.2)
    except Exception as e:
        failed.append((r.cell_id, repr(e)[:80]))
    if i % 25 == 0: print(f"  {i}/{len(panel)} ok={done} fail={len(failed)}")

daily = pd.concat([pd.read_parquet(f) for f in sorted(RAW.glob("*.parquet"))], ignore_index=True)
daily.to_parquet(OUT/"power_daily_panel.parquet", index=False)

print("\n--- SANITY ---")
print("cells:", daily.cell_id.nunique(), "of", len(panel), "| failures:", len(failed))
if failed: print("  first:", failed[:3])
n = daily.groupby("cell_id").size()
print("rows/cell: min %d max %d (must be equal)" % (n.min(), n.max()))
print("GWETTOP: %.3f - %.3f in [0,1]? %s"
      % (daily.GWETTOP.min(), daily.GWETTOP.max(), bool(daily.GWETTOP.between(0,1).all())))
gaps = {c:int(gg.date.diff().dt.days.fillna(1).ne(1).sum()) for c,gg in daily.groupby("cell_id")}
print("cells with date gaps:", sum(v>0 for v in gaps.values()), "(must be 0)")
j = panel.set_index("cell_id").join(daily.groupby("cell_id").GWETTOP.mean().rename("gw"))
print("\nGWETTOP by aridity (must INCREASE):")
print(j.groupby("aridity_class", observed=True).gw.agg(["count","mean"]).round(3).to_string())
print("*** corr(AI, GWETTOP) = %.3f  (want clearly positive)" % j[["AI","gw"]].corr().iloc[0,1])

  0/250 ok=1 fail=0
  25/250 ok=26 fail=0
  50/250 ok=51 fail=0
  75/250 ok=76 fail=0
  100/250 ok=101 fail=0
  125/250 ok=126 fail=0
  150/250 ok=151 fail=0
  175/250 ok=176 fail=0
  200/250 ok=201 fail=0
  225/250 ok=226 fail=0

--- SANITY ---
cells: 250 of 250 | failures: 0
rows/cell: min 16436 max 16436 (must be equal)
GWETTOP: 0.020 - 1.000 in [0,1]? True
cells with date gaps: 0 (must be 0)

GWETTOP by aridity (must INCREASE):
               count   mean
aridity_class              
hyperarid         36  0.186
arid              52  0.321
semiarid          54  0.454
dry-subhumid      54  0.508
humid             54  0.687
*** corr(AI, GWETTOP) = 0.799  (want clearly positive)


## Cell 3 — monthly panel + Stage A diagnostics

In [ ]:
from statsmodels.tsa.stattools import acf, pacf

daily = pd.read_parquet(OUT/"power_daily_panel.parquet")
panel = pd.read_parquet(OUT/"panel_global.parquet")
daily["date"] = pd.to_datetime(daily.date)

AGG = {"GWETTOP":"mean","GWETROOT":"mean","GWETPROF":"mean","PRECTOTCORR":"sum",
       "T2M":"mean","T2M_MAX":"mean","T2M_MIN":"mean","RH2M":"mean","WS2M":"mean",
       "ALLSKY_SFC_SW_DWN":"mean","EVLAND":"mean","PS":"mean"}
mon = (daily.set_index("date").groupby(["cell_id","name"]).resample("MS").agg(AGG)
             .reset_index().rename(columns={"date":"month"}))
mon = mon[mon.month >= "1984-01-01"].reset_index(drop=True)
mon.to_parquet(OUT/"power_monthly_panel.parquet", index=False)

def harmonics(t, n_h=2, period=12):
    return np.column_stack([f(2*np.pi*k*t/period) for k in range(1,n_h+1) for f in (np.sin,np.cos)])

def sampen(x, m=2, r=0.2):
    x = (x-x.mean())/x.std(); N = len(x)
    def phi(mm):
        X = np.array([x[i:i+mm] for i in range(N-mm+1)])
        D = np.max(np.abs(X[:,None,:]-X[None,:,:]), axis=2); np.fill_diagonal(D, np.inf)
        return (D <= r).sum()
    A, B = phi(m+1), phi(m)
    return np.inf if A == 0 else -np.log(A/B)

rows = []
for cid, gg in mon.groupby("cell_id"):
    gg = gg.sort_values("month"); y = gg.GWETTOP.to_numpy(float); t = np.arange(len(y))
    H = np.column_stack([np.ones(len(y)), harmonics(t)])
    b, *_ = np.linalg.lstsq(H, y, rcond=None); resid = y - H@b
    a_raw, a_des = acf(y, nlags=24, fft=True), acf(resid, nlags=24, fft=True)
    p_des = pacf(resid, nlags=12, method="ywadjusted")
    rows.append(dict(cell_id=cid, n_months=len(y), mean=y.mean(), sd=y.std(),
        seas_share=1-resid.var()/y.var(), acf1_raw=a_raw[1], acf12_raw=a_raw[12],
        acf1_des=a_des[1], acf2_des=a_des[2], acf12_des=a_des[12],
        pacf1_des=p_des[1], sampen_des=sampen(resid), resid_sd=resid.std()))

diag = pd.DataFrame(rows).merge(panel[["cell_id","name","lat","lon","AI","aridity_class",
    "irrig_index","irrig_class","hemi","replication"]], on="cell_id")
diag.to_parquet(OUT/"stageA_diagnostics.parquet", index=False)

print("--- SANITY ---")
print("monthly rows:", len(mon), "| months/cell:", mon.groupby("cell_id").size().unique())
print("GWETTOP in [0,1]?", bool(mon.GWETTOP.between(0,1).all()))
print("seas_share %.3f-%.3f | acf1_des %.3f-%.3f"
      % (diag.seas_share.min(),diag.seas_share.max(),diag.acf1_des.min(),diag.acf1_des.max()))
print("\nby aridity:\n", diag.groupby("aridity_class",observed=True)[["seas_share","acf1_des","sampen_des"]].mean().round(3).to_string())
print("\nby hemisphere:\n", diag.groupby("hemi")[["seas_share","acf1_des"]].mean().round(3).to_string())
print("\nPunjab:\n", diag[diag.replication][["name","seas_share","acf1_des","sampen_des"]].round(3).to_string(index=False))

/usr/local/lib/python3.13/dist-packages/statsmodels/tsa/stattools.py:702: RuntimeWarning: invalid value encountered in divide
  acf = avf[: nlags + 1] / avf[0]
/usr/local/lib/python3.13/dist-packages/statsmodels/regression/linear_model.py:1490: ValueWarning: Matrix is singular. Using pinv.
  warnings.warn("Matrix is singular. Using pinv.", ValueWarning)
/tmp/ipykernel_2238/3420688255.py:35: RuntimeWarning: invalid value encountered in scalar divide
  seas_share=1-resid.var()/y.var(), acf1_raw=a_raw[1], acf12_raw=a_raw[12],
/tmp/ipykernel_2238/3420688255.py:19: RuntimeWarning: invalid value encountered in divide
  x = (x-x.mean())/x.std(); N = len(x)


--- SANITY ---
monthly rows: 126000 | months/cell: [504]
GWETTOP in [0,1]? True
seas_share 0.057-0.944 | acf1_des 0.373-0.952

by aridity:
                seas_share  acf1_des  sampen_des
aridity_class                                  
hyperarid           0.429     0.632       1.261
arid                0.583     0.670       1.550
semiarid            0.568     0.698       1.597
dry-subhumid        0.581     0.718       1.621
humid               0.467     0.789         inf

by hemisphere:
       seas_share  acf1_des
hemi                      
N          0.556     0.706
S          0.505     0.707

Punjab:
       name  seas_share  acf1_des  sampen_des
  Firozpur       0.443     0.679       1.863
      Moga       0.476     0.702       1.722
Hoshiarpur       0.531     0.738       1.629


In [ ]:

# DIAGNOSTIC — locate degenerate series
diag = pd.read_parquet(OUT/"stageA_diagnostics.parquet")
mon  = pd.read_parquet(OUT/"power_monthly_panel.parquet")

v = mon.groupby("cell_id").GWETTOP.agg(["mean","std","min","max","nunique"])
bad = v[(v["std"] < 0.005) | (v.nunique < 20)]
print("degenerate cells:", len(bad))
print(bad.round(4).to_string())

print("\nnon-finite diagnostics:")
for c in ["seas_share","acf1_des","sampen_des"]:
    n = (~np.isfinite(diag[c])).sum()
    print(f"  {c}: {n}")
    if n: print(diag.loc[~np.isfinite(diag[c]), ["cell_id","AI","aridity_class","mean","sd"]].to_string(index=False))

print("\nsaturated (mean > 0.98):", int((v["mean"] > 0.98).sum()))
print("lowest std cells:\n", v.nsmallest(8, "std").round(4).to_string())

TypeError: '<' not supported between instances of 'method' and 'int'

In [ ]:

# DIAGNOSTIC — locate degenerate series
import numpy as np, pandas as pd
diag = pd.read_parquet(OUT/"stageA_diagnostics.parquet")
mon  = pd.read_parquet(OUT/"power_monthly_panel.parquet")

v = mon.groupby("cell_id").GWETTOP.agg(["mean","std","min","max","nunique"])
bad = v[(v["std"] < 0.005) | (v["nunique"] < 20)]
print("degenerate cells:", len(bad))
if len(bad): print(bad.round(4).to_string())

print("\nnon-finite diagnostics:")
for c in ["seas_share","acf1_des","sampen_des"]:
    m = ~np.isfinite(diag[c])
    print(f"  {c}: {int(m.sum())}")
    if m.any():
        print(diag.loc[m, ["cell_id","AI","aridity_class","mean","sd"]].round(4).to_string(index=False))

print("\nsaturated (mean > 0.98):", int((v["mean"] > 0.98).sum()))
print("\nlowest-variance cells:\n", v.nsmallest(8, "std").round(4).to_string())

degenerate cells: 1
                  mean  std  min  max  nunique
cell_id                                       
-07p500_-034p688   1.0  0.0  1.0  1.0        1

non-finite diagnostics:
  seas_share: 1
         cell_id     AI aridity_class  mean  sd
-07p500_-034p688 1.1568         humid   1.0 0.0
  acf1_des: 1
         cell_id     AI aridity_class  mean  sd
-07p500_-034p688 1.1568         humid   1.0 0.0
  sampen_des: 1
         cell_id     AI aridity_class  mean  sd
-07p500_-034p688 1.1568         humid   1.0 0.0

saturated (mean > 0.98): 1

lowest-variance cells:
                     mean     std     min     max  nunique
cell_id                                                  
-07p500_-034p688  1.0000  0.0000  1.0000  1.0000        1
-30p250_+030p938  0.8787  0.0236  0.8148  0.9463      340
-07p000_+157p188  0.9583  0.0261  0.7390  0.9847      288
-53p750_-067p812  0.8801  0.0268  0.8118  0.9390      349
-50p750_-069p062  0.6539  0.0289  0.5975  0.7455      374
+25p500_+032p812  0.0

In [ ]:

# Drop degenerate (zero-variance) cells and rebuild Stage A
import numpy as np, pandas as pd
from pathlib import Path
OUT = Path("data")

mon   = pd.read_parquet(OUT/"power_monthly_panel.parquet")
panel = pd.read_parquet(OUT/"panel_global.parquet")
diag  = pd.read_parquet(OUT/"stageA_diagnostics.parquet")

MIN_SD = 0.01
sd_ = mon.groupby("cell_id").GWETTOP.std()
drop = sd_[sd_ < MIN_SD].index.tolist()
print("dropping", len(drop), "degenerate cell(s):", drop)
print(panel[panel.cell_id.isin(drop)][["cell_id","lat","lon","AI","aridity_class"]].to_string(index=False))

mon   = mon[~mon.cell_id.isin(drop)].reset_index(drop=True)
panel = panel[~panel.cell_id.isin(drop)].reset_index(drop=True)
diag  = diag[~diag.cell_id.isin(drop)].reset_index(drop=True)

mon.to_parquet(OUT/"power_monthly_panel.parquet", index=False)
panel.to_parquet(OUT/"panel_global.parquet", index=False)
diag.to_parquet(OUT/"stageA_diagnostics.parquet", index=False)

print("\n--- SANITY ---")
print("panel now:", len(panel), "cells | monthly rows:", len(mon))
for c in ["seas_share","acf1_des","sampen_des"]:
    print(f"  non-finite {c}: {int((~np.isfinite(diag[c])).sum())} (must be 0)")
print("min GWETTOP sd across cells: %.4f" % mon.groupby('cell_id').GWETTOP.std().min())
print("\nby aridity:\n", diag.groupby("aridity_class",observed=True)[["seas_share","acf1_des","sampen_des"]].mean().round(3).to_string())

dropping 1 degenerate cell(s): ['-07p500_-034p688']
         cell_id  lat      lon       AI aridity_class
-07p500_-034p688 -7.5 -34.6875 1.156842         humid

--- SANITY ---
panel now: 249 cells | monthly rows: 125496
  non-finite seas_share: 0 (must be 0)
  non-finite acf1_des: 0 (must be 0)
  non-finite sampen_des: 0 (must be 0)
min GWETTOP sd across cells: 0.0236

by aridity:
                seas_share  acf1_des  sampen_des
aridity_class                                  
hyperarid           0.429     0.632       1.261
arid                0.583     0.670       1.550
semiarid            0.568     0.698       1.597
dry-subhumid        0.581     0.718       1.621
humid               0.467     0.789       1.404


## Cell 4 — Stage B: predictability ceilings

Linear (Toeplitz / Yule–Walker) and information (KSG) bounds. **No confidence interval** — block
bootstrapping a seasonal series requires phase- and amplitude-preserving resampling we did not
implement, and the naive version is biased low in proportion to seasonal share. The point
estimator is validated against closed-form AR(2) solutions (error < 0.003; see Cell 4v).

In [ ]:
from scipy.linalg import toeplitz, solve_toeplitz
from scipy.special import digamma
from sklearn.neighbors import NearestNeighbors

mon  = pd.read_parquet(OUT/"power_monthly_panel.parquet")
diag = pd.read_parquet(OUT/"stageA_diagnostics.parquet")
LAGS_LIN, LAGS_MI = 12, [1,2,12]

def deseason(y, n_h=2):
    t = np.arange(len(y))
    H = np.column_stack([np.ones(len(y))] +
        [f(2*np.pi*k*t/12) for k in range(1,n_h+1) for f in (np.sin,np.cos)])
    b, *_ = np.linalg.lstsq(H, y, rcond=None); return y - H@b

def ceiling_linear(y, p=LAGS_LIN):
    gam = acf(y, nlags=p, fft=True) * y.var()
    try: w = solve_toeplitz((gam[:p], gam[:p]), gam[1:p+1])
    except Exception: w = np.linalg.lstsq(toeplitz(gam[:p]), gam[1:p+1], rcond=None)[0]
    return float(np.clip(1 - (gam[0] - w@gam[1:p+1])/gam[0], 0, 1))

def ksg_mi(X, y, k=4):
    X = (X-X.mean(0))/X.std(0); y = (y-y.mean())/y.std()
    Z = np.column_stack([X, y]); n = len(Z)
    eps = NearestNeighbors(n_neighbors=k+1, metric="chebyshev").fit(Z).kneighbors(Z)[0][:,k]
    def cnt(A):
        nn = NearestNeighbors(metric="chebyshev").fit(A)
        return np.array([len(nn.radius_neighbors(A[j:j+1], radius=eps[j]-1e-12,
                         return_distance=False)[0])-1 for j in range(n)])
    return max(digamma(k)+digamma(n)-np.mean(digamma(cnt(X)+1)+digamma(cnt(y[:,None])+1)), 0.0)

def lagmat(y, lags):
    m = max(lags); return np.column_stack([y[m-L:len(y)-L] for L in lags]), y[m:]

rows = []
for ci,(cid, gg) in enumerate(mon.groupby("cell_id")):
    y = gg.sort_values("month").GWETTOP.to_numpy(float); yd = deseason(y)
    Xr, Yr = lagmat(y, LAGS_MI); Xd, Yd = lagmat(yd, LAGS_MI)
    mi_r, mi_d = ksg_mi(Xr, Yr), ksg_mi(Xd, Yd)
    rows.append(dict(cell_id=cid, ceil_lin_raw=ceiling_linear(y), ceil_lin_des=ceiling_linear(yd),
        mi_raw=mi_r, mi_des=mi_d,
        ceil_mi_raw=float(np.clip(1-np.exp(-2*mi_r),0,1)),
        ceil_mi_des=float(np.clip(1-np.exp(-2*mi_d),0,1))))
    if ci % 50 == 0: print(f"  cell {ci}")

ceil = pd.DataFrame(rows).merge(diag[["cell_id","name","AI","aridity_class","irrig_index",
    "irrig_class","hemi","seas_share","acf1_des","sampen_des","replication"]], on="cell_id")
ceil.to_parquet(OUT/"stageB_ceilings.parquet", index=False)

print("\n--- SANITY ---")
for c in ["ceil_lin_raw","ceil_lin_des","ceil_mi_raw","ceil_mi_des"]:
    print(f"{c}: {ceil[c].min():.3f}-{ceil[c].max():.3f} in [0,1]? {bool(ceil[c].between(0,1).all())}")
print("raw >= des everywhere?", bool((ceil.ceil_lin_raw >= ceil.ceil_lin_des-1e-9).all()))
print("panel mean ceiling: %.3f" % ceil.ceil_lin_raw.mean())
print("\nby aridity:\n", ceil.groupby("aridity_class",observed=True)[["ceil_lin_raw","ceil_lin_des","seas_share"]].mean().round(3).to_string())
print("\ncorr(ceil_lin_des, acf1_des)   = %.3f" % ceil[["ceil_lin_des","acf1_des"]].corr().iloc[0,1])
print("corr(ceil_lin_raw, seas_share) = %.3f" % ceil[["ceil_lin_raw","seas_share"]].corr().iloc[0,1])
print("\nPunjab:\n", ceil[ceil.replication][["name","ceil_lin_raw","ceil_lin_des","ceil_mi_raw"]].round(3).to_string(index=False))

  cell 0
  cell 50
  cell 100
  cell 150
  cell 200

--- SANITY ---
ceil_lin_raw: 0.272-0.951 in [0,1]? True
ceil_lin_des: 0.166-0.909 in [0,1]? True
ceil_mi_raw: 0.288-1.000 in [0,1]? True
ceil_mi_des: 0.204-1.000 in [0,1]? True
raw >= des everywhere? False
panel mean ceiling: 0.751

by aridity:
                ceil_lin_raw  ceil_lin_des  seas_share
aridity_class                                        
hyperarid             0.637         0.437       0.429
arid                  0.742         0.480       0.583
semiarid              0.754         0.527       0.568
dry-subhumid          0.775         0.551       0.581
humid                 0.811         0.656       0.467

corr(ceil_lin_des, acf1_des)   = 0.984
corr(ceil_lin_raw, seas_share) = 0.721

Punjab:
       name  ceil_lin_raw  ceil_lin_des  ceil_mi_raw
  Firozpur         0.627         0.474        0.605
      Moga         0.664         0.510        0.655
Hoshiarpur         0.734         0.571        0.729


In [ ]:

# DIAGNOSTIC — ceiling anomalies
import numpy as np, pandas as pd
ceil = pd.read_parquet(OUT/"stageB_ceilings.parquet")
mon  = pd.read_parquet(OUT/"power_monthly_panel.parquet")
sd_  = mon.groupby("cell_id").GWETTOP.std().rename("gw_sd")
c = ceil.merge(sd_, on="cell_id")

inv = c[c.ceil_lin_raw < c.ceil_lin_des - 1e-9]
print("cells with raw < des:", len(inv))
if len(inv):
    print(inv[["cell_id","aridity_class","gw_sd","seas_share","acf1_des",
               "ceil_lin_raw","ceil_lin_des"]].round(4).to_string(index=False))
    print("  max violation: %.4f" % (inv.ceil_lin_des - inv.ceil_lin_raw).max())

sat = c[(c.ceil_mi_raw > 0.995) | (c.ceil_mi_des > 0.995)]
print("\nMI bound saturated (>0.995):", len(sat))
if len(sat):
    print(sat[["cell_id","aridity_class","gw_sd","mi_raw","mi_des",
               "ceil_mi_raw","ceil_lin_raw"]].round(4).to_string(index=False))

print("\ncorr(gw_sd, mi_raw) = %.3f" % c[["gw_sd","mi_raw"]].corr().iloc[0,1])
print("lowest-variance cells:\n",
      c.nsmallest(6,"gw_sd")[["cell_id","gw_sd","ceil_lin_raw","ceil_lin_des","ceil_mi_raw"]].round(4).to_string(index=False))

cells with raw < des: 12
         cell_id aridity_class  gw_sd  seas_share  acf1_des  ceil_lin_raw  ceil_lin_des
+00p000_+031p562         humid 0.0888      0.1233    0.9140        0.8393        0.8427
+00p000_+035p938         humid 0.1076      0.1963    0.8524        0.7454        0.7552
+00p000_+130p938         humid 0.0546      0.0655    0.8708        0.7708        0.7734
+02p000_-077p188         humid 0.1049      0.2059    0.9323        0.8678        0.8742
+06p000_+080p938         humid 0.0752      0.1313    0.9296        0.8519        0.8690
+06p000_+125p312         humid 0.1449      0.1158    0.9101        0.8367        0.8370
+07p500_+125p938         humid 0.0985      0.0595    0.9329        0.8720        0.8762
+43p000_+113p438          arid 0.0562      0.1718    0.8270        0.6936        0.6966
+43p500_+123p438      semiarid 0.0540      0.0977    0.7707        0.6346        0.6626
+46p000_+124p688  dry-subhumid 0.0614      0.0828    0.8464        0.7198        0.7320
-00p500

In [ ]:

# Fix KSG MI on tied (floor-clipped) series; leave linear ceilings untouched
import numpy as np, pandas as pd
from pathlib import Path
from scipy.special import digamma
from sklearn.neighbors import NearestNeighbors
OUT = Path("data")
rng_j = np.random.default_rng(0)

ceil = pd.read_parquet(OUT/"stageB_ceilings.parquet")
mon  = pd.read_parquet(OUT/"power_monthly_panel.parquet")

def ksg_mi_j(X, y, k=4, jitter=1e-8):
    X = (X-X.mean(0))/(X.std(0)+1e-12); y = (y-y.mean())/(y.std()+1e-12)
    X = X + rng_j.normal(0, jitter, X.shape)      # break exact ties
    y = y + rng_j.normal(0, jitter, y.shape)
    Z = np.column_stack([X, y]); n = len(Z)
    eps = NearestNeighbors(n_neighbors=k+1, metric="chebyshev").fit(Z).kneighbors(Z)[0][:,k]
    eps = np.maximum(eps, 1e-12)
    def cnt(A):
        nn = NearestNeighbors(metric="chebyshev").fit(A)
        return np.array([len(nn.radius_neighbors(A[j:j+1], radius=eps[j]*(1-1e-9),
                         return_distance=False)[0])-1 for j in range(n)])
    nx, ny = np.maximum(cnt(X),0), np.maximum(cnt(y[:,None]),0)
    mi = digamma(k) + digamma(n) - np.mean(digamma(nx+1) + digamma(ny+1))
    return float(np.clip(mi, 0.0, 5.0))           # 5 nats -> R2 bound 0.99995

def deseason(y, n_h=2):
    t = np.arange(len(y))
    H = np.column_stack([np.ones(len(y))] +
        [f(2*np.pi*k*t/12) for k in range(1,n_h+1) for f in (np.sin,np.cos)])
    b,*_ = np.linalg.lstsq(H, y, rcond=None); return y - H@b

def lagmat(y, lags):
    m = max(lags); return np.column_stack([y[m-L:len(y)-L] for L in lags]), y[m:]

LAGS_MI = [1,2,12]
rows = []
for cid, gg in mon.groupby("cell_id"):
    y  = gg.sort_values("month").GWETTOP.to_numpy(float); yd = deseason(y)
    Xr,Yr = lagmat(y,LAGS_MI); Xd,Yd = lagmat(yd,LAGS_MI)
    mr, md = ksg_mi_j(Xr,Yr), ksg_mi_j(Xd,Yd)
    rows.append(dict(cell_id=cid, mi_raw=mr, mi_des=md,
                     ceil_mi_raw=float(np.clip(1-np.exp(-2*mr),0,1)),
                     ceil_mi_des=float(np.clip(1-np.exp(-2*md),0,1))))
fix = pd.DataFrame(rows)

ceil = ceil.drop(columns=["mi_raw","mi_des","ceil_mi_raw","ceil_mi_des"]).merge(fix, on="cell_id")
ceil.to_parquet(OUT/"stageB_ceilings.parquet", index=False)

print("--- SANITY ---")
print("non-finite mi:", int((~np.isfinite(ceil.mi_raw)).sum()), "(must be 0)")
print("ceil_mi_raw > 0.995:", int((ceil.ceil_mi_raw > 0.995).sum()), "(was 5)")
print("ceil_mi_raw: %.3f - %.3f" % (ceil.ceil_mi_raw.min(), ceil.ceil_mi_raw.max()))
print("corr(ceil_mi_raw, ceil_lin_raw) = %.3f  (want high; they should agree)"
      % ceil[["ceil_mi_raw","ceil_lin_raw"]].corr().iloc[0,1])
print("\nhyperarid cells:\n",
      ceil[ceil.aridity_class=="hyperarid"][["cell_id","mi_raw","ceil_mi_raw","ceil_lin_raw"]].round(3).head(8).to_string(index=False))

--- SANITY ---
non-finite mi: 0 (must be 0)
ceil_mi_raw > 0.995: 0 (was 5)
ceil_mi_raw: 0.286 - 0.951
corr(ceil_mi_raw, ceil_lin_raw) = 0.937  (want high; they should agree)

hyperarid cells:
          cell_id  mi_raw  ceil_mi_raw  ceil_lin_raw
+10p500_+045p312   0.323        0.476         0.531
+16p000_+028p438   0.890        0.831         0.610
+18p000_-015p938   1.049        0.877         0.824
+25p500_+032p812   0.637        0.720         0.593
+27p500_-114p062   0.419        0.568         0.527
+28p500_+030p938   0.929        0.844         0.686
+28p500_+067p812   0.292        0.443         0.543
+28p500_+068p438   0.253        0.397         0.496


## Cell 4v — validate the ceiling estimator against analytic AR(2)

In [ ]:
rng_v = np.random.default_rng(0)
print("AR(2) validation — true R^2 from closed form vs estimator:")
for (a1,a2,sig) in [(0.6,0.2,1.0),(0.9,-0.3,1.0),(0.3,0.0,1.0)]:
    n=100000; e=rng_v.normal(0,sig,n); y=np.zeros(n)
    for t in range(2,n): y[t]=a1*y[t-1]+a2*y[t-2]+e[t]
    y=y[1000:]; true_r2=1-sig**2/y.var(); est=ceiling_linear(y)
    print(f"  AR2({a1},{a2}): true {true_r2:.4f} | est {est:.4f} | diff {est-true_r2:+.4f}")
print("\n-> all |diff| should be < 0.005")

AR(2) validation — true R^2 from closed form vs estimator:
  AR2(0.6,0.2): true 0.5786 | est 0.5783 | diff -0.0003
  AR2(0.9,-0.3): true 0.5247 | est 0.5223 | diff -0.0024
  AR2(0.3,0.0): true 0.0916 | est 0.0901 | diff -0.0016

-> all |diff| should be < 0.005


In [ ]:

# RESTORE SESSION
import numpy as np, pandas as pd, warnings
from pathlib import Path
from scipy import stats
from statsmodels.tsa.stattools import acf, pacf
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
warnings.filterwarnings("ignore")
OUT = Path("data")
try:
    from xgboost import XGBRegressor; HAS_XGB = True
except ImportError: HAS_XGB = False
try:
    from lightgbm import LGBMRegressor; HAS_LGB = True
except ImportError: HAS_LGB = False

mon   = pd.read_parquet(OUT/"power_monthly_panel.parquet")
panel = pd.read_parquet(OUT/"panel_global.parquet")
diag  = pd.read_parquet(OUT/"stageA_diagnostics.parquet")
ceil  = pd.read_parquet(OUT/"stageB_ceilings.parquet")

LAGS = list(range(1,13)); N_FOLDS, TEST_LEN, SEEDS = 5, 60, 5
def nse(o,p): return 1 - ((o-p)**2).sum()/((o-o.mean())**2).sum()
def kge(o,p):
    r = np.corrcoef(o,p)[0,1]
    return 1 - np.sqrt((r-1)**2 + (p.std()/o.std()-1)**2 + (p.mean()/o.mean()-1)**2)
def design(y, months):
    m = max(LAGS)
    return np.column_stack([y[m-L:len(y)-L] for L in LAGS]), y[m:], months[m:]

ck = OUT/"stageC_ck"
print("cells:", mon.cell_id.nunique(),
      "| stageC done:", len(list(ck.glob("*.parquet"))) if ck.exists() else 0, "of 249")

FileNotFoundError: [Errno 2] No such file or directory: 'data/power_monthly_panel.parquet'

In [ ]:

from pathlib import Path
import shutil
from google.colab import files
Path("data").mkdir(exist_ok=True)
up = files.upload()                       # ssw_global_results.zip
shutil.unpack_archive("ssw_global_results (2).zip", "data")
print(sorted(p.name for p in Path("data").glob("*")))

Saving ssw_global_results (2).zip to ssw_global_results (2) (2).zip
['drive-download-20260822T163700Z-1-001.zip', 'gpc_0.csv', 'gpc_1.csv', 'gpc_10.csv', 'gpc_11.csv', 'gpc_12.csv', 'gpc_13.csv', 'gpc_14.csv', 'gpc_15.csv', 'gpc_16.csv', 'gpc_17.csv', 'gpc_18.csv', 'gpc_19.csv', 'gpc_2.csv', 'gpc_20.csv', 'gpc_21.csv', 'gpc_22.csv', 'gpc_23.csv', 'gpc_3.csv', 'gpc_4.csv', 'gpc_5.csv', 'gpc_6.csv', 'gpc_7.csv', 'gpc_8.csv', 'gpc_9.csv', 'panel_global.parquet', 'power_daily_panel.parquet', 'power_monthly_panel.parquet', 'raw_daily', 'stageA_diagnostics.parquet', 'stageB_ceilings.parquet']


## Cell 5 — Stage C: benchmark ladder (checkpointed)

`SEEDS = 5` by default; 250 cells × 5 folds × 6 models × 5 seeds. Budget 40–90 min.
Raise to 10 for the final run. **Run Cell Z after this.**

In [ ]:
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
import warnings; warnings.filterwarnings("ignore")
try:
    from xgboost import XGBRegressor; HAS_XGB = True
except ImportError: HAS_XGB = False
try:
    from lightgbm import LGBMRegressor; HAS_LGB = True
except ImportError: HAS_LGB = False

CK_C = OUT/"stageC_ck"; CK_C.mkdir(exist_ok=True)
mon  = pd.read_parquet(OUT/"power_monthly_panel.parquet")
ceil = pd.read_parquet(OUT/"stageB_ceilings.parquet")
LAGS = list(range(1,13)); N_FOLDS, TEST_LEN, SEEDS = 5, 60, 5

def nse(o,p): return 1 - ((o-p)**2).sum()/((o-o.mean())**2).sum()
def kge(o,p):
    r = np.corrcoef(o,p)[0,1]
    return 1 - np.sqrt((r-1)**2 + (p.std()/o.std()-1)**2 + (p.mean()/o.mean()-1)**2)
def design(y, months):
    m = max(LAGS)
    return np.column_stack([y[m-L:len(y)-L] for L in LAGS]), y[m:], months[m:]

def make_models(seed):
    M = {"AR12": LinearRegression(),
         "CART": DecisionTreeRegressor(random_state=seed),
         "RF": RandomForestRegressor(n_estimators=300, min_samples_leaf=4, random_state=seed, n_jobs=-1),
         "AdaBoost-CART": AdaBoostRegressor(DecisionTreeRegressor(max_depth=6, random_state=seed),
                                            n_estimators=100, random_state=seed)}
    if HAS_XGB: M["XGBoost"] = XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.05,
                                            subsample=0.8, random_state=seed, verbosity=0)
    if HAS_LGB: M["LightGBM"] = LGBMRegressor(n_estimators=300, num_leaves=31,
                                              learning_rate=0.05, random_state=seed, verbose=-1)
    return M

for ci,(cid, gg) in enumerate(mon.groupby("cell_id")):
    fpk = CK_C/f"{cid}.parquet"
    if fpk.exists(): continue
    gg = gg.sort_values("month")
    X, Y, Mo = design(gg.GWETTOP.to_numpy(float), gg.month.to_numpy()); n = len(Y)
    rr = []
    for f in range(N_FOLDS):
        te_hi = n-(N_FOLDS-1-f)*TEST_LEN; te_lo = te_hi-TEST_LEN
        if te_lo < 120: continue
        Xtr,Ytr,Xte,Yte = X[:te_lo],Y[:te_lo],X[te_lo:te_hi],Y[te_lo:te_hi]
        mo_tr = pd.DatetimeIndex(Mo[:te_lo]).month; mo_te = pd.DatetimeIndex(Mo[te_lo:te_hi]).month
        clim = pd.Series(Ytr).groupby(mo_tr).mean()
        for nm,p in {"Climatology":clim.reindex(mo_te).to_numpy(),
                     "Persistence":Xte[:,0], "SeasonalNaive":Xte[:,11]}.items():
            rr.append(dict(cell_id=cid,fold=f,model=nm,seed=0,nse=nse(Yte,p),kge=kge(Yte,p)))
        for s in range(SEEDS):
            for nm,mdl in make_models(s).items():
                if nm=="AR12" and s>0: continue
                mdl.fit(Xtr,Ytr); p=mdl.predict(Xte)
                rr.append(dict(cell_id=cid,fold=f,model=nm,seed=s,nse=nse(Yte,p),kge=kge(Yte,p)))
    pd.DataFrame(rr).to_parquet(fpk, index=False)
    if ci % 25 == 0: print(f"  cell {ci}/{mon.cell_id.nunique()}")

res = pd.concat([pd.read_parquet(f) for f in sorted(CK_C.glob("*.parquet"))], ignore_index=True)
res.to_parquet(OUT/"stageC_legacy_results.parquet", index=False)

print("\n--- SANITY ---")
print("cells:", res.cell_id.nunique(), "| rows:", len(res), "| non-finite:", int((~np.isfinite(res.nse)).sum()))
print("\nmean NSE by model:\n", res.groupby("model").nse.agg(["mean","std"]).round(3).sort_values("mean",ascending=False).to_string())
cb = res.groupby(["cell_id","model"]).nse.mean().reset_index()
top = cb.loc[cb.groupby("cell_id").nse.idxmax()]
print("\nwins by cell:\n", top.model.value_counts().to_string())
ml = res[~res.model.isin(["Climatology","Persistence","SeasonalNaive","AR12"])]
seed_sd = ml.groupby(["cell_id","fold","model"]).nse.std().dropna().median()
betw_sd = ml.groupby(["cell_id","fold"]).apply(lambda z: z.groupby("model").nse.mean().std()).dropna().median()
print("\n*** seed sd %.4f | between-model sd %.4f | ratio %.1fx" % (seed_sd, betw_sd, betw_sd/seed_sd))
t = top.merge(ceil, on="cell_id"); t["gap"] = t.ceil_lin_raw - t.nse
t.to_parquet(OUT/"stageC_vs_ceiling.parquet", index=False)
print("*** corr(best NSE, ceiling) = %.3f | mean gap %.3f | above ceiling %d/%d"
      % (t[["nse","ceil_lin_raw"]].corr().iloc[0,1], t.gap.mean(), int((t.gap<0).sum()), len(t)))

  cell 0/249
  cell 25/249
  cell 50/249
  cell 75/249
  cell 100/249
  cell 125/249
  cell 150/249
  cell 175/249
  cell 200/249
  cell 225/249

--- SANITY ---
cells: 249 | rows: 36105 | non-finite: 0

mean NSE by model:
                 mean    std
model                      
AR12           0.687  0.210
RF             0.662  0.227
AdaBoost-CART  0.650  0.244
XGBoost        0.633  0.269
LightGBM       0.616  0.293
Persistence    0.490  0.213
Climatology    0.354  0.787
CART           0.348  0.500
SeasonalNaive  0.125  0.596

wins by cell:
 model
AR12             167
RF                53
AdaBoost-CART     16
XGBoost            6
LightGBM           4
Climatology        2
Persistence        1

*** seed sd 0.0070 | between-model sd 0.0992 | ratio 14.1x
*** corr(best NSE, ceiling) = 0.937 | mean gap 0.058 | above ceiling 48/249


In [ ]:

import shutil
from google.colab import files
shutil.make_archive("ssw_global_stageC", "zip", "data")
files.download("ssw_global_stageC.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# Where does AR12 lose, and to what?
res  = pd.read_parquet(OUT/"stageC_legacy_results.parquet")
ceil = pd.read_parquet(OUT/"stageB_ceilings.parquet")
cb   = res.groupby(["cell_id","model"]).nse.mean().reset_index()
top  = cb.loc[cb.groupby("cell_id").nse.idxmax()].merge(ceil, on="cell_id")
top["ar12_wins"] = top.model == "AR12"

print("win rate by aridity class:")
print(top.groupby("aridity_class", observed=True).ar12_wins.agg(["count","mean"]).round(3).to_string())
print("\nwinner x aridity:\n", pd.crosstab(top.aridity_class, top.model).to_string())
print("\nMI bound above linear bound?  (nonlinear structure present)")
top["mi_gt_lin"] = top.ceil_mi_raw > top.ceil_lin_raw
print(pd.crosstab(top.mi_gt_lin, top.ar12_wins).to_string())
print("\nAR12 win rate where MI > linear: %.3f | where MI <= linear: %.3f"
      % (top[top.mi_gt_lin].ar12_wins.mean(), top[~top.mi_gt_lin].ar12_wins.mean()))
print("\nclimatology outliers:\n",
      res[res.model=="Climatology"].groupby("cell_id").nse.mean().nsmallest(5).round(3).to_string())

win rate by aridity class:
               count   mean
aridity_class              
hyperarid         36  0.722
arid              52  0.558
semiarid          54  0.630
dry-subhumid      54  0.593
humid             53  0.868

winner x aridity:
 model          AR12  AdaBoost-CART  Climatology  LightGBM  Persistence  RF  XGBoost
aridity_class                                                                      
hyperarid        26              1            1         0            0   6        2
arid             29              4            0         0            1  17        1
semiarid         34              3            0         1            0  15        1
dry-subhumid     32              7            1         2            0  12        0
humid            46              1            0         1            0   3        2

MI bound above linear bound?  (nonlinear structure present)
ar12_wins  False  True 
mi_gt_lin              
False         33     95
True          49     72

AR12 win ra

## Cell 6 — tuning-free tier: ESN (+ optional Chronos zero-shot)

The ESN readout is 301-dimensional; inputs and target are standardised and the ridge penalty is
selected on an inner holdout. A weak fixed ridge diverges — do not remove this.

Set `RUN_CHRONOS = True` on a GPU runtime for the zero-shot foundation-model comparison.

In [ ]:
RUN_CHRONOS = False    # True after: !pip install -q chronos-forecasting  (use a GPU runtime)
ESN_SEEDS = 5
CK_E = OUT/"stageE_ck"; CK_E.mkdir(exist_ok=True)

class ESN:
    def __init__(self, n_res=300, rho=0.9, sparsity=0.1, leak=0.3,
                 ridge=None, in_scale=0.5, seed=0):
        self.p=dict(n_res=n_res,rho=rho,leak=leak,in_scale=in_scale)
        self.ridge=ridge; self.sparsity=sparsity; self.seed=seed
    def _init(self,n_in):
        N=self.p["n_res"]; r=np.random.default_rng(self.seed)
        W=r.normal(0,1,(N,N))*(r.random((N,N))<self.sparsity)
        ev=np.max(np.abs(np.linalg.eigvals(W)))
        self.W=W*(self.p["rho"]/ev if ev>0 else 1.0)
        self.Win=r.uniform(-0.5,0.5,(N,n_in))*self.p["in_scale"]
    def _states(self,X):
        N,a=self.p["n_res"],self.p["leak"]; s=np.zeros(N); S=np.empty((len(X),N))
        for i,x in enumerate(X):
            s=(1-a)*s+a*np.tanh(self.Win@x+self.W@s); S[i]=s
        return S
    def _readout(self,S,y,lam):
        A=np.column_stack([S,np.ones(len(S))])
        return np.linalg.solve(A.T@A+lam*np.eye(A.shape[1]),A.T@y)
    def fit(self,X,y):
        self.mu,self.sd=X.mean(0),X.std(0)+1e-9; self.ym,self.ys=y.mean(),y.std()+1e-9
        Xs,ys_=(X-self.mu)/self.sd,(y-self.ym)/self.ys
        self._init(Xs.shape[1]); S=self._states(Xs)
        if self.ridge is None:
            cut=int(len(S)*0.8); best,blam=-np.inf,1.0
            for lam in [1e-2,1e-1,1.0,10.0,100.0]:
                w=self._readout(S[:cut],ys_[:cut],lam)
                p=np.column_stack([S[cut:],np.ones(len(S)-cut)])@w
                sc=-np.mean((p-ys_[cut:])**2)
                if sc>best: best,blam=sc,lam
            self.lam_=blam
        else: self.lam_=self.ridge
        self.Wout=self._readout(S,ys_,self.lam_); return self
    def predict(self,X):
        S=self._states((X-self.mu)/self.sd)
        return (np.column_stack([S,np.ones(len(S))])@self.Wout)*self.ys+self.ym

pipe=None
if RUN_CHRONOS:
    import torch
    from chronos import BaseChronosPipeline
    dev="cuda" if torch.cuda.is_available() else "cpu"
    pipe=BaseChronosPipeline.from_pretrained("amazon/chronos-bolt-base",
                                             device_map=dev,torch_dtype=torch.float32)
    print("chronos on",dev)

for ci,(cid,gg) in enumerate(mon.groupby("cell_id")):
    fpk=CK_E/f"{cid}.parquet"
    if fpk.exists(): continue
    gg=gg.sort_values("month"); y=gg.GWETTOP.to_numpy(float)
    X,Y,_=design(y,gg.month.to_numpy()); m0=max(LAGS); n=len(Y); rr=[]
    for f in range(N_FOLDS):
        te_hi=n-(N_FOLDS-1-f)*TEST_LEN; te_lo=te_hi-TEST_LEN
        if te_lo<120: continue
        Xtr,Ytr,Xte,Yte=X[:te_lo],Y[:te_lo],X[te_lo:te_hi],Y[te_lo:te_hi]
        for s in range(ESN_SEEDS):
            p=ESN(seed=s).fit(Xtr,Ytr).predict(Xte)
            rr.append(dict(cell_id=cid,fold=f,model="ESN",seed=s,nse=nse(Yte,p),kge=kge(Yte,p)))
        if pipe is not None:
            ctx=[torch.tensor(y[:m0+te_lo+t],dtype=torch.float32) for t in range(len(Yte))]
            q,_m=pipe.predict_quantiles(context=ctx,prediction_length=1,quantile_levels=[0.5])
            p=q[:,0,0].numpy()
            rr.append(dict(cell_id=cid,fold=f,model="Chronos-Bolt(0shot)",seed=0,
                           nse=nse(Yte,p),kge=kge(Yte,p)))
    pd.DataFrame(rr).to_parquet(fpk,index=False)
    if ci%25==0: print(f"  cell {ci}")

new=pd.concat([pd.read_parquet(f) for f in sorted(CK_E.glob("*.parquet"))],ignore_index=True)
res=pd.concat([pd.read_parquet(OUT/"stageC_legacy_results.parquet"),new],ignore_index=True)
res.to_parquet(OUT/"stageC_all_results.parquet",index=False)

print("\n--- SANITY ---")
esn=res[res.model=="ESN"]
print("ESN mean %.3f | min %.3f | frac<0 %.2f" % (esn.nse.mean(),esn.nse.min(),(esn.nse<0).mean()))
print("  -> if mean is large-negative or min < -5, the reservoir diverged; DO NOT report")
print("\nmean NSE:\n", res.groupby("model").nse.agg(["mean","std"]).round(3).sort_values("mean",ascending=False).to_string())
tuned=res[res.model.isin(["RF","XGBoost","LightGBM"])].nse.mean()
free =res[res.model.isin(["ESN","Chronos-Bolt(0shot)"])].nse.mean()
print("\ntuned tree tier %.3f | tuning-free tier %.3f | diff %+.3f" % (tuned,free,tuned-free))
print("AR12 %.3f | climatology floor %.3f"
      % (res[res.model=="AR12"].nse.mean(), res[res.model=="Climatology"].nse.mean()))

  cell 0
  cell 25
  cell 50
  cell 75
  cell 100
  cell 125
  cell 150
  cell 175
  cell 200
  cell 225

--- SANITY ---
ESN mean 0.634 | min -2.147 | frac<0 0.03
  -> if mean is large-negative or min < -5, the reservoir diverged; DO NOT report

mean NSE:
                 mean    std
model                      
AR12           0.687  0.210
RF             0.662  0.227
AdaBoost-CART  0.650  0.244
ESN            0.634  0.267
XGBoost        0.633  0.269
LightGBM       0.616  0.293
Persistence    0.490  0.213
Climatology    0.354  0.787
CART           0.348  0.500
SeasonalNaive  0.125  0.596

tuned tree tier 0.637 | tuning-free tier 0.634 | diff +0.002
AR12 0.687 | climatology floor 0.354


In [ ]:

import shutil
from google.colab import files
shutil.make_archive("ssw_global_stageC_esn", "zip", "data")
files.download("ssw_global_stageC_esn.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 7 — Stage D (PRIMARY): drivers at lags 1–3

Forecast-realistic. This is the headline driver result.

In [ ]:
DRV = ["PRECTOTCORR","T2M","T2M_MAX","RH2M","WS2M","ALLSKY_SFC_SW_DWN"]

def run_stageD(drv_lags, ck_name, out_name):
    CK = OUT/ck_name; CK.mkdir(exist_ok=True)
    def build(gg):
        y=gg.GWETTOP.to_numpy(float); m=max(max(LAGS),max(drv_lags))
        Xlag=np.column_stack([y[m-L:len(y)-L] for L in LAGS])
        cols=[]
        for d in DRV:
            v=gg[d].to_numpy(float)
            for L in drv_lags: cols.append(v[m-L:len(v)-L])
        return Xlag,np.column_stack(cols),y[m:]
    def dmodels(seed):
        M={"AR/OLS":LinearRegression(),
           "RF":RandomForestRegressor(n_estimators=300,min_samples_leaf=4,random_state=seed,n_jobs=-1)}
        if HAS_LGB: M["LightGBM"]=LGBMRegressor(n_estimators=300,num_leaves=31,
                                                learning_rate=0.05,random_state=seed,verbose=-1)
        return M
    for ci,(cid,gg) in enumerate(mon.groupby("cell_id")):
        fpk=CK/f"{cid}.parquet"
        if fpk.exists(): continue
        gg=gg.sort_values("month"); Xlag,Xdrv,Y=build(gg); n=len(Y)
        SETS={"lag_only":Xlag,"lag+drivers":np.column_stack([Xlag,Xdrv]),"drivers_only":Xdrv}
        rr=[]
        for f in range(N_FOLDS):
            te_hi=n-(N_FOLDS-1-f)*TEST_LEN; te_lo=te_hi-TEST_LEN
            if te_lo<120: continue
            for sn,XX in SETS.items():
                for s in range(SEEDS):
                    for mn,mdl in dmodels(s).items():
                        if mn=="AR/OLS" and s>0: continue
                        mdl.fit(XX[:te_lo],Y[:te_lo]); p=mdl.predict(XX[te_lo:te_hi])
                        rr.append(dict(cell_id=cid,fold=f,inputs=sn,model=mn,seed=s,
                                       nse=nse(Y[te_lo:te_hi],p)))
        pd.DataFrame(rr).to_parquet(fpk,index=False)
        if ci%25==0: print(f"  cell {ci}")
    d=pd.concat([pd.read_parquet(f) for f in sorted(CK.glob("*.parquet"))],ignore_index=True)
    best=d.groupby(["cell_id","inputs","model"]).nse.mean().reset_index()
    best=best.loc[best.groupby(["cell_id","inputs"]).nse.idxmax()]
    piv=best.pivot(index="cell_id",columns="inputs",values="nse").join(
        ceil.set_index("cell_id")[["name","ceil_lin_raw","acf1_des","sampen_des","seas_share",
                                   "irrig_class","aridity_class","hemi","replication"]])
    piv["driver_gain"]=piv["lag+drivers"]-piv["lag_only"]
    piv["above_ceiling"]=piv["lag+drivers"]-piv["ceil_lin_raw"]
    piv.to_parquet(OUT/out_name)
    return d,piv

d1,sd1 = run_stageD([1,2,3], "stageD_ck_lag1", "stageD_summary_lag1.parquet")

print("\n--- SANITY (FORECAST-HONEST, lags 1-3) ---")
print("cells:", d1.cell_id.nunique())
print("mean NSE by input set:\n", d1.groupby("inputs").nse.mean().round(3).to_string())
print("\nmean driver gain: %.3f" % sd1.driver_gain.mean())
print("Wilcoxon: p = %.2g" % stats.wilcoxon(sd1["lag+drivers"], sd1["lag_only"])[1])
print("cells above lag-only ceiling: %d / %d" % (int((sd1.above_ceiling>0).sum()), len(sd1)))
for v in ["acf1_des","sampen_des"]:
    r_,p_ = stats.pearsonr(sd1[v].astype(float), sd1.driver_gain.astype(float))
    print("corr(%-11s, driver_gain) = %+.3f (p=%.2g)" % (v,r_,p_))
print("\nby aridity:\n", sd1.groupby("aridity_class",observed=True)[["lag_only","lag+drivers","driver_gain"]].mean().round(3).to_string())
print("\nPunjab:\n", sd1[sd1.replication][["name","lag_only","lag+drivers","driver_gain","ceil_lin_raw","above_ceiling"]].round(3).to_string())

  cell 0
  cell 25
  cell 50
  cell 75
  cell 100
  cell 125
  cell 150
  cell 175
  cell 200
  cell 225

--- SANITY (FORECAST-HONEST, lags 1-3) ---
cells: 249
mean NSE by input set:
 inputs
drivers_only    0.664
lag+drivers     0.714
lag_only        0.643

mean driver gain: 0.047
Wilcoxon: p = 1.3e-39
cells above lag-only ceiling: 137 / 249
corr(acf1_des   , driver_gain) = -0.353 (p=1e-08)
corr(sampen_des , driver_gain) = +0.204 (p=0.0012)

by aridity:
                lag_only  lag+drivers  driver_gain
aridity_class                                    
hyperarid         0.544        0.611        0.067
arid              0.690        0.743        0.053
semiarid          0.710        0.755        0.045
dry-subhumid      0.728        0.769        0.041
humid             0.742        0.780        0.038

Punjab:
                         name  lag_only  lag+drivers  driver_gain  ceil_lin_raw  above_ceiling
cell_id                                                                                

In [ ]:

import shutil
from google.colab import files
shutil.make_archive("ssw_global_stageD1", "zip", "data")
files.download("ssw_global_stageD1.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 7b — Stage D (DIAGNOSTIC): drivers at lags 0–2

Includes the concurrent month. Reported as an information-content diagnostic **only** — this
configuration recovers SSW from the meteorology that generates it inside the same land-surface
model, and is not a forecast.

In [ ]:
d0,sd0 = run_stageD([0,1,2], "stageD_ck_lag0", "stageD_summary_lag0.parquet")

print("\n--- SANITY (DIAGNOSTIC, lags 0-2) ---")
print("mean NSE by input set:\n", d0.groupby("inputs").nse.mean().round(3).to_string())
print("\n*** drivers_only here should be MUCH higher than in Cell 7 — that gap is the leakage")
print("mean driver gain: %.3f  (honest variant gave %.3f)" % (sd0.driver_gain.mean(), sd1.driver_gain.mean()))
print("cells above ceiling: %d/%d  (honest gave %d/%d)"
      % (int((sd0.above_ceiling>0).sum()), len(sd0), int((sd1.above_ceiling>0).sum()), len(sd1)))
r0,p0 = stats.pearsonr(sd0.acf1_des.astype(float), sd0.driver_gain.astype(float))
r1,p1 = stats.pearsonr(sd1.acf1_des.astype(float), sd1.driver_gain.astype(float))
print("\ncorr(acf1_des, driver_gain): leaked %+.3f (p=%.2g) | honest %+.3f (p=%.2g)" % (r0,p0,r1,p1))
print("  -> if the leaked version is significant and the honest one is not, that IS the finding")

NameError: name 'run_stageD' is not defined

In [ ]:

# CELL 7b (self-contained) — Stage D DIAGNOSTIC, drivers at lags 0-2
import numpy as np, pandas as pd, warnings
from pathlib import Path
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
warnings.filterwarnings("ignore")
OUT = Path("data")
try:
    from lightgbm import LGBMRegressor; HAS_LGB = True
except ImportError: HAS_LGB = False

mon  = pd.read_parquet(OUT/"power_monthly_panel.parquet")
ceil = pd.read_parquet(OUT/"stageB_ceilings.parquet")
LAGS = list(range(1,13)); N_FOLDS, TEST_LEN, SEEDS = 5, 60, 5
DRV  = ["PRECTOTCORR","T2M","T2M_MAX","RH2M","WS2M","ALLSKY_SFC_SW_DWN"]

def nse(o,p): return 1 - ((o-p)**2).sum()/((o-o.mean())**2).sum()

def run_stageD(drv_lags, ck_name, out_name):
    CK = OUT/ck_name; CK.mkdir(exist_ok=True)
    def build(gg):
        y = gg.GWETTOP.to_numpy(float); m = max(max(LAGS), max(drv_lags))
        Xlag = np.column_stack([y[m-L:len(y)-L] for L in LAGS])
        cols = []
        for d in DRV:
            v = gg[d].to_numpy(float)
            for L in drv_lags: cols.append(v[m-L:len(v)-L])
        return Xlag, np.column_stack(cols), y[m:]
    def dmodels(seed):
        M = {"AR/OLS": LinearRegression(),
             "RF": RandomForestRegressor(n_estimators=300, min_samples_leaf=4,
                                         random_state=seed, n_jobs=-1)}
        if HAS_LGB: M["LightGBM"] = LGBMRegressor(n_estimators=300, num_leaves=31,
                                                  learning_rate=0.05, random_state=seed, verbose=-1)
        return M
    for ci,(cid, gg) in enumerate(mon.groupby("cell_id")):
        fpk = CK/f"{cid}.parquet"
        if fpk.exists(): continue
        gg = gg.sort_values("month"); Xlag, Xdrv, Y = build(gg); n = len(Y)
        SETS = {"lag_only":Xlag, "lag+drivers":np.column_stack([Xlag,Xdrv]), "drivers_only":Xdrv}
        rr = []
        for f in range(N_FOLDS):
            te_hi = n-(N_FOLDS-1-f)*TEST_LEN; te_lo = te_hi-TEST_LEN
            if te_lo < 120: continue
            for sn, XX in SETS.items():
                for s in range(SEEDS):
                    for mn, mdl in dmodels(s).items():
                        if mn == "AR/OLS" and s > 0: continue
                        mdl.fit(XX[:te_lo], Y[:te_lo]); p = mdl.predict(XX[te_lo:te_hi])
                        rr.append(dict(cell_id=cid, fold=f, inputs=sn, model=mn, seed=s,
                                       nse=nse(Y[te_lo:te_hi], p)))
        pd.DataFrame(rr).to_parquet(fpk, index=False)
        if ci % 25 == 0: print(f"  cell {ci}")
    d = pd.concat([pd.read_parquet(f) for f in sorted(CK.glob("*.parquet"))], ignore_index=True)
    best = d.groupby(["cell_id","inputs","model"]).nse.mean().reset_index()
    best = best.loc[best.groupby(["cell_id","inputs"]).nse.idxmax()]
    piv = best.pivot(index="cell_id", columns="inputs", values="nse").join(
          ceil.set_index("cell_id")[["name","ceil_lin_raw","acf1_des","sampen_des","seas_share",
                                     "irrig_class","aridity_class","hemi","replication"]])
    piv["driver_gain"]   = piv["lag+drivers"] - piv["lag_only"]
    piv["above_ceiling"] = piv["lag+drivers"] - piv["ceil_lin_raw"]
    piv.to_parquet(OUT/out_name)
    return d, piv

d0, sd0 = run_stageD([0,1,2], "stageD_ck_lag0", "stageD_summary_lag0.parquet")
sd1 = pd.read_parquet(OUT/"stageD_summary_lag1.parquet")

print("\n--- SANITY (DIAGNOSTIC, lags 0-2) ---")
print("mean NSE by input set:\n", d0.groupby("inputs").nse.mean().round(3).to_string())
print("\n*** drivers_only here vs 0.664 in the honest run — that gap IS the leakage")
print("driver gain: %.3f  (honest gave 0.047)" % sd0.driver_gain.mean())
print("above ceiling: %d/249  (honest gave 137/249)" % int((sd0.above_ceiling>0).sum()))
r0,p0 = stats.pearsonr(sd0.acf1_des.astype(float), sd0.driver_gain.astype(float))
r1,p1 = stats.pearsonr(sd1.acf1_des.astype(float), sd1.driver_gain.astype(float))
print("\ncorr(acf1_des, gain): leaked %+.3f (p=%.2g) | honest %+.3f (p=%.2g)" % (r0,p0,r1,p1))
print("inflation factor: %.1fx" % (sd0.driver_gain.mean()/sd1.driver_gain.mean()))

KeyboardInterrupt: 

In [ ]:

from pathlib import Path
import shutil
from google.colab import files
Path("data").mkdir(exist_ok=True)
up = files.upload()                      # ssw_global_stageD1.zip
shutil.unpack_archive("ssw_global_stageD1.zip", "data")
print(sorted(p.name for p in Path("data").glob("*")))

Saving ssw_global_stageD1.zip to ssw_global_stageD1.zip
['drive-download-20260822T163700Z-1-001.zip', 'gpc_0.csv', 'gpc_1.csv', 'gpc_10.csv', 'gpc_11.csv', 'gpc_12.csv', 'gpc_13.csv', 'gpc_14.csv', 'gpc_15.csv', 'gpc_16.csv', 'gpc_17.csv', 'gpc_18.csv', 'gpc_19.csv', 'gpc_2.csv', 'gpc_20.csv', 'gpc_21.csv', 'gpc_22.csv', 'gpc_23.csv', 'gpc_3.csv', 'gpc_4.csv', 'gpc_5.csv', 'gpc_6.csv', 'gpc_7.csv', 'gpc_8.csv', 'gpc_9.csv', 'panel_global.parquet', 'power_daily_panel.parquet', 'power_monthly_panel.parquet', 'raw_daily', 'stageA_diagnostics.parquet', 'stageB_ceilings.parquet', 'stageC_all_results.parquet', 'stageC_ck', 'stageC_legacy_results.parquet', 'stageC_vs_ceiling.parquet', 'stageD_ck_lag0', 'stageD_ck_lag1', 'stageD_summary_lag1.parquet', 'stageE_ck']


In [ ]:

# CELL 7b (self-contained) — Stage D DIAGNOSTIC, drivers at lags 0-2
import numpy as np, pandas as pd, warnings
from pathlib import Path
from scipy import stats
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
warnings.filterwarnings("ignore")
OUT = Path("data")
try:
    from lightgbm import LGBMRegressor; HAS_LGB = True
except ImportError: HAS_LGB = False

mon  = pd.read_parquet(OUT/"power_monthly_panel.parquet")
ceil = pd.read_parquet(OUT/"stageB_ceilings.parquet")
LAGS = list(range(1,13)); N_FOLDS, TEST_LEN, SEEDS = 5, 60, 5
DRV  = ["PRECTOTCORR","T2M","T2M_MAX","RH2M","WS2M","ALLSKY_SFC_SW_DWN"]

def nse(o,p): return 1 - ((o-p)**2).sum()/((o-o.mean())**2).sum()

def run_stageD(drv_lags, ck_name, out_name):
    CK = OUT/ck_name; CK.mkdir(exist_ok=True)
    def build(gg):
        y = gg.GWETTOP.to_numpy(float); m = max(max(LAGS), max(drv_lags))
        Xlag = np.column_stack([y[m-L:len(y)-L] for L in LAGS])
        cols = []
        for d in DRV:
            v = gg[d].to_numpy(float)
            for L in drv_lags: cols.append(v[m-L:len(v)-L])
        return Xlag, np.column_stack(cols), y[m:]
    def dmodels(seed):
        M = {"AR/OLS": LinearRegression(),
             "RF": RandomForestRegressor(n_estimators=300, min_samples_leaf=4,
                                         random_state=seed, n_jobs=-1)}
        if HAS_LGB: M["LightGBM"] = LGBMRegressor(n_estimators=300, num_leaves=31,
                                                  learning_rate=0.05, random_state=seed, verbose=-1)
        return M
    for ci,(cid, gg) in enumerate(mon.groupby("cell_id")):
        fpk = CK/f"{cid}.parquet"
        if fpk.exists(): continue
        gg = gg.sort_values("month"); Xlag, Xdrv, Y = build(gg); n = len(Y)
        SETS = {"lag_only":Xlag, "lag+drivers":np.column_stack([Xlag,Xdrv]), "drivers_only":Xdrv}
        rr = []
        for f in range(N_FOLDS):
            te_hi = n-(N_FOLDS-1-f)*TEST_LEN; te_lo = te_hi-TEST_LEN
            if te_lo < 120: continue
            for sn, XX in SETS.items():
                for s in range(SEEDS):
                    for mn, mdl in dmodels(s).items():
                        if mn == "AR/OLS" and s > 0: continue
                        mdl.fit(XX[:te_lo], Y[:te_lo]); p = mdl.predict(XX[te_lo:te_hi])
                        rr.append(dict(cell_id=cid, fold=f, inputs=sn, model=mn, seed=s,
                                       nse=nse(Y[te_lo:te_hi], p)))
        pd.DataFrame(rr).to_parquet(fpk, index=False)
        if ci % 25 == 0: print(f"  cell {ci}")
    d = pd.concat([pd.read_parquet(f) for f in sorted(CK.glob("*.parquet"))], ignore_index=True)
    best = d.groupby(["cell_id","inputs","model"]).nse.mean().reset_index()
    best = best.loc[best.groupby(["cell_id","inputs"]).nse.idxmax()]
    piv = best.pivot(index="cell_id", columns="inputs", values="nse").join(
          ceil.set_index("cell_id")[["name","ceil_lin_raw","acf1_des","sampen_des","seas_share",
                                     "irrig_class","aridity_class","hemi","replication"]])
    piv["driver_gain"]   = piv["lag+drivers"] - piv["lag_only"]
    piv["above_ceiling"] = piv["lag+drivers"] - piv["ceil_lin_raw"]
    piv.to_parquet(OUT/out_name)
    return d, piv

d0, sd0 = run_stageD([0,1,2], "stageD_ck_lag0", "stageD_summary_lag0.parquet")
sd1 = pd.read_parquet(OUT/"stageD_summary_lag1.parquet")

print("\n--- SANITY (DIAGNOSTIC, lags 0-2) ---")
print("mean NSE by input set:\n", d0.groupby("inputs").nse.mean().round(3).to_string())
print("\n*** drivers_only here vs 0.664 in the honest run — that gap IS the leakage")
print("driver gain: %.3f  (honest gave 0.047)" % sd0.driver_gain.mean())
print("above ceiling: %d/249  (honest gave 137/249)" % int((sd0.above_ceiling>0).sum()))
r0,p0 = stats.pearsonr(sd0.acf1_des.astype(float), sd0.driver_gain.astype(float))
r1,p1 = stats.pearsonr(sd1.acf1_des.astype(float), sd1.driver_gain.astype(float))
print("\ncorr(acf1_des, gain): leaked %+.3f (p=%.2g) | honest %+.3f (p=%.2g)" % (r0,p0,r1,p1))
print("inflation factor: %.1fx" % (sd0.driver_gain.mean()/sd1.driver_gain.mean()))

  cell 225

--- SANITY (DIAGNOSTIC, lags 0-2) ---
mean NSE by input set:
 inputs
drivers_only    0.859
lag+drivers     0.904
lag_only        0.643

*** drivers_only here vs 0.664 in the honest run — that gap IS the leakage
driver gain: 0.238  (honest gave 0.047)
above ceiling: 246/249  (honest gave 137/249)

corr(acf1_des, gain): leaked -0.302 (p=1.2e-06) | honest -0.353 (p=1e-08)
inflation factor: 5.0x


In [ ]:

import shutil
from google.colab import files
shutil.make_archive("ssw_global_complete", "zip", "data")
files.download("ssw_global_complete.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:

# UNIVERSAL RESTORE — upload any project zip, unpack into data/
from pathlib import Path
import shutil, zipfile
OUT = Path("data"); OUT.mkdir(exist_ok=True)

from google.colab import files
up = files.upload()                       # pick your .zip

for name in up:
    if not name.lower().endswith(".zip"):
        (OUT/name).write_bytes(up[name]); continue
    with zipfile.ZipFile(name) as z:
        z.extractall(OUT)
    print(f"unpacked {name}: {len(z.namelist())} entries")

# flatten any nested folders so data/ is one level deep
for p in list(OUT.rglob("*")):
    if p.is_file() and p.parent != OUT:
        tgt = OUT/p.name
        if not tgt.exists(): p.rename(tgt)
for p in sorted(OUT.rglob("*"), reverse=True):
    if p.is_dir() and not any(p.iterdir()): p.rmdir()

print("\n--- data/ ---")
for p in sorted(OUT.glob("*")):
    if p.is_dir():
        print(f"  {p.name}/  ({len(list(p.glob('*')))} files)")
    else:
        print(f"  {p.name}  {p.stat().st_size//1024} KB")

KeyboardInterrupt: 

In [3]:

# RESTORE, preserving checkpoint folders
from pathlib import Path
import zipfile
OUT = Path("data"); OUT.mkdir(exist_ok=True)
from google.colab import files
up = files.upload()

for name in up:
    if name.lower().endswith(".zip"):
        with zipfile.ZipFile(name) as z: z.extractall(OUT)
        print(f"unpacked {name}")

print("\n--- data/ ---")
for p in sorted(OUT.glob("*")):
    print(f"  {p.name}/ ({len(list(p.glob('*')))} files)" if p.is_dir()
          else f"  {p.name}  {p.stat().st_size//1024} KB")
print("\nkey files present:")
for f in ["panel_global.parquet","power_monthly_panel.parquet","stageB_ceilings.parquet",
          "stageC_all_results.parquet","stageD_summary_lag1.parquet","stageD_summary_lag0.parquet"]:
    print(f"  {'OK ' if (OUT/f).exists() else 'MISSING'} {f}")

Saving ssw_global_complete.zip to ssw_global_complete (1).zip
unpacked ssw_global_complete (1).zip

--- data/ ---
  drive-download-20260822T163700Z-1-001.zip  2325 KB
  gpc_0.csv  0 KB
  gpc_1.csv  0 KB
  gpc_10.csv  142 KB
  gpc_11.csv  185 KB
  gpc_12.csv  88 KB
  gpc_13.csv  337 KB
  gpc_14.csv  573 KB
  gpc_15.csv  461 KB
  gpc_16.csv  28 KB
  gpc_17.csv  63 KB
  gpc_18.csv  578 KB
  gpc_19.csv  570 KB
  gpc_2.csv  13 KB
  gpc_20.csv  174 KB
  gpc_21.csv  100 KB
  gpc_22.csv  71 KB
  gpc_23.csv  330 KB
  gpc_3.csv  220 KB
  gpc_4.csv  124 KB
  gpc_5.csv  175 KB
  gpc_6.csv  345 KB
  gpc_7.csv  521 KB
  gpc_8.csv  71 KB
  gpc_9.csv  186 KB
  panel_global.parquet  28 KB
  power_daily_panel.parquet  67163 KB
  power_monthly_panel.parquet  7491 KB
  raw_daily/ (250 files)
  stageA_diagnostics.parquet  44 KB
  stageB_ceilings.parquet  35 KB
  stageC_all_results.parquet  758 KB
  stageC_ck/ (249 files)
  stageC_legacy_results.parquet  626 KB
  stageC_vs_ceiling.parquet  41 KB
  stageD_ck

In [4]:

# Re-render figures at 300 dpi, no titles, Fig 4 as densities
import matplotlib.pyplot as plt, numpy as np, pandas as pd
from pathlib import Path
from scipy import stats
OUT, FIG = Path("data"), Path("figs300"); FIG.mkdir(exist_ok=True)
plt.rcParams.update({"font.size":9,"font.family":"serif","savefig.bbox":"tight","savefig.dpi":300})

ceil = pd.read_parquet(OUT/"stageB_ceilings.parquet")
res  = pd.read_parquet(OUT/"stageC_all_results.parquet")
pan  = pd.read_parquet(OUT/"panel_global.parquet")
sd1  = pd.read_parquet(OUT/"stageD_summary_lag1.parquet")
sd0  = pd.read_parquet(OUT/"stageD_summary_lag0.parquet")

cl = ceil.merge(pan[["cell_id","lat","lon"]], on="cell_id")
f,a = plt.subplots(figsize=(7.2,3.6))
sc = a.scatter(cl.lon, cl.lat, c=cl.AI, s=14+90*cl.irrig_index.fillna(0), cmap="BrBG",
               vmin=0, vmax=1.5, edgecolor="k", linewidth=.3)
plt.colorbar(sc,label="aridity index $P/PET$")
a.set_xlabel("longitude (°E)"); a.set_ylabel("latitude (°N)")
a.set_xlim(-180,180); a.set_ylim(-60,80)
f.savefig(FIG/"fig1_global_panel.png"); plt.close(f)

best = res.groupby(["cell_id","model"]).nse.mean().reset_index()
top  = best.loc[best.groupby("cell_id").nse.idxmax()].merge(ceil, on="cell_id")
f,a = plt.subplots(figsize=(4.0,3.8))
for c_,g in top.groupby("aridity_class",observed=True): a.scatter(g.ceil_lin_raw,g.nse,s=13,label=c_,alpha=.85)
lim=[0,1]; a.plot(lim,lim,"k--",lw=1,label="1:1")
a.set_xlabel("linear predictability ceiling"); a.set_ylabel("best achieved NSE"); a.legend(fontsize=7,frameon=False)
f.savefig(FIG/"fig2_ceiling_vs_achieved.png"); plt.close(f)

order = res.groupby("model").nse.mean().sort_values().index.tolist()
sub = res.merge(ceil[["cell_id","aridity_class"]], on="cell_id")
f,a = plt.subplots(figsize=(6.0,3.6))
for c_ in [x for x in ["hyperarid","arid","semiarid","dry-subhumid","humid"] if x in sub.aridity_class.unique()]:
    m_=sub[sub.aridity_class==c_].groupby("model").nse.mean().reindex(order)
    a.plot(range(len(order)),m_.values,"o-",ms=3.5,lw=1.1,label=c_)
    a.axhline(ceil[ceil.aridity_class==c_].ceil_lin_raw.mean(),ls=":",lw=.8,color=a.lines[-1].get_color())
a.set_xticks(range(len(order))); a.set_xticklabels(order,rotation=40,ha="right")
a.set_ylabel("mean NSE"); a.legend(fontsize=7,frameon=False,title="dotted = class ceiling",title_fontsize=7)
f.savefig(FIG/"fig3_ladder.png"); plt.close(f)

ml = res[~res.model.isin(["Climatology","Persistence","SeasonalNaive","AR12"])]
seed_sd = ml.groupby(["cell_id","fold","model"]).nse.std().dropna()
betw_sd = ml.groupby(["cell_id","fold"]).apply(lambda z: z.groupby("model").nse.mean().std()).dropna()
f,a = plt.subplots(figsize=(4.4,3.4))
bins = np.linspace(0,0.35,60)
a.hist(seed_sd,bins=bins,density=True,alpha=.6,label=f"within-model, across seeds (med {seed_sd.median():.4f})")
a.hist(betw_sd,bins=bins,density=True,alpha=.6,label=f"between-model (med {betw_sd.median():.4f})")
a.axvline(seed_sd.median(),color="C0",ls="--",lw=1); a.axvline(betw_sd.median(),color="C1",ls="--",lw=1)
a.set_xlabel("s.d. of NSE"); a.set_ylabel("density"); a.legend(fontsize=7,frameon=False)
f.savefig(FIG/"fig4_seed_vs_model.png"); plt.close(f)

f,axes = plt.subplots(1,2,figsize=(7.0,3.3),sharey=True)
for ax,s_,ttl in [(axes[0],sd0,"(a) drivers incl. concurrent month"),
                  (axes[1],sd1,"(b) drivers lagged 1–3 months")]:
    for c_,g in s_.groupby("irrig_class",observed=True): ax.scatter(g.acf1_des,g.driver_gain,s=12,label=c_,alpha=.8)
    x,y=s_.acf1_des.astype(float),s_.driver_gain.astype(float); ok=x.notna()&y.notna()
    sl,ic,r_,p_,_=stats.linregress(x[ok],y[ok]); xx=np.linspace(x[ok].min(),x[ok].max(),50)
    ax.plot(xx,sl*xx+ic,"k-",lw=1.1,label=f"r = {r_:.2f}")
    ax.axhline(0,color="grey",lw=.7); ax.set_xlabel("deseasonalised lag-1 autocorrelation")
    ax.set_title(ttl,fontsize=8.5); ax.legend(fontsize=6.5,frameon=False)
axes[0].set_ylabel("NSE gain from drivers")
f.savefig(FIG/"fig5_driver_gain_both.png"); plt.close(f)

f,a = plt.subplots(figsize=(4.4,3.4))
s_=sd1.sort_values("above_ceiling")
a.barh(range(len(s_)),s_.above_ceiling,color=["#b2434a" if v<0 else "#3d6d9e" for v in s_.above_ceiling],height=1.0)
a.axvline(0,color="k",lw=.9)
a.set_xlabel("NSE with lagged drivers  −  lag-only ceiling")
a.set_ylabel(f"cells (n = {len(s_)}, sorted)"); a.set_yticks([])
f.savefig(FIG/"fig6_exceedance.png"); plt.close(f)

import shutil
shutil.make_archive("figs300","zip","figs300")
try:
    from google.colab import files; files.download("figs300.zip")
except ImportError: pass
for p in sorted(FIG.glob("*.png")): print(p.name, p.stat().st_size//1024, "KB")

/tmp/ipykernel_479/2117522632.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  betw_sd = ml.groupby(["cell_id","fold"]).apply(lambda z: z.groupby("model").nse.mean().std()).dropna()


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

fig1_global_panel.png 201 KB
fig2_ceiling_vs_achieved.png 165 KB
fig3_ladder.png 204 KB
fig4_seed_vs_model.png 61 KB
fig5_driver_gain_both.png 251 KB
fig6_exceedance.png 37 KB


In [5]:

import requests
dois = ["10.1038/sdata.2017.191","10.5067/MODIS/MCD12Q1.006","10.1175/JCLI-D-16-0758.1",
        "10.1016/j.rse.2017.06.031","10.1038/s41598-026-50687-9","10.1103/PhysRevE.69.066138",
        "10.1016/0022-1694(70)90255-6","10.1016/j.earscirev.2010.02.004","10.1016/j.rse.2018.12.013"]
for d in dois:
    try:
        r = requests.get(f"https://api.crossref.org/works/{d}", timeout=15)
        if r.ok:
            m = r.json()["message"]
            print(f"OK   {d}\n     {m.get('container-title',[''])[0]} {m.get('volume','')} ({m.get('issued',{}).get('date-parts',[['?']])[0][0]})")
        else:
            print(f"FAIL {d}  HTTP {r.status_code}")
    except Exception as e:
        print(f"ERR  {d}  {e}")

OK   10.1038/sdata.2017.191
     Scientific Data 5 (2018)
FAIL 10.5067/MODIS/MCD12Q1.006  HTTP 404
OK   10.1175/JCLI-D-16-0758.1
     Journal of Climate 30 (2017)
OK   10.1016/j.rse.2017.06.031
     Remote Sensing of Environment 202 (2017)
OK   10.1038/s41598-026-50687-9
     Scientific Reports 16 (2026)
OK   10.1103/PhysRevE.69.066138
     Physical Review E 69 (2004)
OK   10.1016/0022-1694(70)90255-6
     Journal of Hydrology 10 (1970)
OK   10.1016/j.earscirev.2010.02.004
     Earth-Science Reviews 99 (2010)
OK   10.1016/j.rse.2018.12.013
     Remote Sensing of Environment 222 (2019)


In [6]:

import requests
r = requests.get("https://api.datacite.org/dois/10.5067/MODIS/MCD12Q1.006", timeout=15)
if r.ok:
    a = r.json()["data"]["attributes"]
    print("OK  ", a.get("titles",[{}])[0].get("title"))
    print("    ", a.get("publisher"), a.get("publicationYear"))
else:
    print("HTTP", r.status_code)

OK   MCD12Q1 MODIS/Terra+Aqua Land Cover Type Yearly L3 Global 500m SIN Grid V006
     NASA Land Processes Distributed Active Archive Center 2019


## Cell 8 — ceiling-exceedance diagnostic

In [ ]:
res  = pd.read_parquet(OUT/"stageC_all_results.parquet")
best = res.groupby(["cell_id","model"]).nse.mean().reset_index()
top  = best.loc[best.groupby("cell_id").nse.idxmax()].merge(ceil, on="cell_id")
top["exceed"] = top.nse - top.ceil_lin_raw
top["over"]   = top.exceed > 0

print("cells above linear ceiling: %d / %d" % (int(top.over.sum()), len(top)))
print("exceedance %+.3f to %+.3f | mean over-cells %+.3f"
      % (top.exceed.min(), top.exceed.max(), top.loc[top.over,"exceed"].mean() if top.over.any() else np.nan))
print("\nover vs not:\n", top.groupby("over")[["seas_share","acf1_des","sampen_des","ceil_lin_raw","AI"]].mean().round(3).to_string())
for v in ["seas_share","acf1_des","sampen_des","AI"]:
    r_,p_ = stats.pearsonr(top[v].astype(float), top.exceed.astype(float))
    print("  corr(%-11s, exceedance) = %+.3f (p=%.3g)" % (v,r_,p_))
print("\nwhich model exceeds (nonlinear should dominate if genuine):\n",
      pd.crosstab(top.model, top.over).to_string())
top.to_parquet(OUT/"exceedance_diagnostic.parquet", index=False)

cells above linear ceiling: 49 / 249
exceedance -0.430 to +0.038 | mean over-cells +0.012

over vs not:
        seas_share  acf1_des  sampen_des  ceil_lin_raw     AI
over                                                        
False       0.475     0.715       1.478         0.729  0.565
True        0.768     0.669       1.605         0.844  0.473
  corr(seas_share , exceedance) = +0.741 (p=1.24e-44)
  corr(acf1_des   , exceedance) = -0.195 (p=0.00204)
  corr(sampen_des , exceedance) = +0.253 (p=5.49e-05)
  corr(AI         , exceedance) = -0.059 (p=0.355)

which model exceeds (nonlinear should dominate if genuine):
 over           False  True 
model                      
AR12             147     13
AdaBoost-CART     10      6
Climatology        0      2
ESN                5      4
LightGBM           0      4
Persistence        1      0
RF                34     17
XGBoost            3      3


In [ ]:

# Largest shortfalls below the ceiling
t = pd.read_parquet(OUT/"exceedance_diagnostic.parquet")
worst = t.nsmallest(8, "exceed")[["cell_id","name","aridity_class","model","nse",
                                  "ceil_lin_raw","exceed","seas_share","acf1_des","sampen_des"]]
print(worst.round(3).to_string(index=False))
print("\ncells with shortfall > 0.20:", int((t.exceed < -0.20).sum()), "of", len(t))
print("their mean sampen: %.3f vs panel %.3f" % (t[t.exceed<-0.20].sampen_des.mean(), t.sampen_des.mean()))

         cell_id name aridity_class model   nse  ceil_lin_raw  exceed  seas_share  acf1_des  sampen_des
-07p000_+157p188 None         humid  AR12 0.073         0.503  -0.430       0.082     0.684       1.320
+17p500_+055p312 None          arid  AR12 0.247         0.629  -0.382       0.068     0.769       0.910
+07p500_+125p938 None         humid  AR12 0.491         0.872  -0.381       0.060     0.933       0.839
-22p250_-068p438 None     hyperarid  AR12 0.333         0.705  -0.372       0.088     0.816       0.789
+43p500_+123p438 None      semiarid  AR12 0.358         0.635  -0.277       0.098     0.771       1.249
+10p500_+045p312 None     hyperarid  AR12 0.263         0.531  -0.268       0.205     0.663       1.813
+46p000_+124p688 None  dry-subhumid  AR12 0.453         0.720  -0.267       0.083     0.846       0.873
+09p000_-064p062 None  dry-subhumid  AR12 0.588         0.836  -0.248       0.421     0.860       1.465

cells with shortfall > 0.20: 12 of 249
their mean sampen: 1.214

In [ ]:

print("shortfall > 0.20:", int((t.exceed<-0.20).sum()),
      "| their mean seas_share %.3f vs panel %.3f" % (t[t.exceed<-0.20].seas_share.mean(), t.seas_share.mean()))
print("corr(seas_share, exceed) among shortfall cells only: %.3f"
      % t[t.exceed<-0.15][["seas_share","exceed"]].corr().iloc[0,1])

shortfall > 0.20: 12 | their mean seas_share 0.157 vs panel 0.532
corr(seas_share, exceed) among shortfall cells only: 0.423


## Cell 9 — figures

In [ ]:
import matplotlib.pyplot as plt
FIG = Path("figs"); FIG.mkdir(exist_ok=True)
plt.rcParams.update({"font.size":9,"figure.dpi":150,"savefig.bbox":"tight"})
sd1 = pd.read_parquet(OUT/"stageD_summary_lag1.parquet")
sd0 = pd.read_parquet(OUT/"stageD_summary_lag0.parquet")
pan = pd.read_parquet(OUT/"panel_global.parquet")

# Fig 1 — global panel map
cl = ceil.merge(pan[["cell_id","lat","lon"]], on="cell_id")
f,a = plt.subplots(figsize=(9,4.5))
sc = a.scatter(cl.lon, cl.lat, c=cl.AI, s=18+120*cl.irrig_index.fillna(0),
               cmap="BrBG", vmin=0, vmax=1.5, edgecolor="k", linewidth=.3)
plt.colorbar(sc,label="aridity index P/PET"); a.set_xlabel("longitude"); a.set_ylabel("latitude")
a.set_xlim(-180,180); a.set_ylim(-60,80)
a.set_title(f"Fig 1  Global panel, n={len(cl)} POWER cells (size = irrigation index)")
f.savefig(FIG/"fig1_global_panel.png"); plt.close(f)

# Fig 2 — achieved vs ceiling
f,a = plt.subplots(figsize=(4.6,4.4))
for c_,gg in top.groupby("aridity_class",observed=True): a.scatter(gg.ceil_lin_raw,gg.nse,s=14,label=c_,alpha=.8)
lim=[0,max(1,top.ceil_lin_raw.max()*1.05)]; a.plot(lim,lim,"k--",lw=1,label="1:1 (at ceiling)")
rr_=np.corrcoef(top.ceil_lin_raw,top.nse)[0,1]
a.set_xlabel("linear predictability ceiling"); a.set_ylabel("best achieved NSE")
a.set_title(f"Fig 2  Skill tracks the ceiling (r = {rr_:.2f})"); a.legend(fontsize=7)
f.savefig(FIG/"fig2_ceiling_vs_achieved.png"); plt.close(f)

# Fig 3 — ladder
order = res.groupby("model").nse.mean().sort_values().index.tolist()
sub = res.merge(ceil[["cell_id","aridity_class"]], on="cell_id")
f,a = plt.subplots(figsize=(6.6,4))
for c_ in [x for x in ["arid","semiarid","dry-subhumid","humid"] if x in sub.aridity_class.unique()]:
    m_=sub[sub.aridity_class==c_].groupby("model").nse.mean().reindex(order)
    a.plot(range(len(order)),m_.values,"o-",ms=4,lw=1.2,label=c_)
    a.axhline(ceil[ceil.aridity_class==c_].ceil_lin_raw.mean(),ls=":",lw=.9,color=a.lines[-1].get_color())
a.set_xticks(range(len(order))); a.set_xticklabels(order,rotation=40,ha="right")
a.set_ylabel("mean NSE"); a.legend(fontsize=7,title="dotted = class ceiling",title_fontsize=7)
a.set_title("Fig 3  Benchmark ladder against the ceiling")
f.savefig(FIG/"fig3_ladder.png"); plt.close(f)

# Fig 4 — seed vs model
ml = res[~res.model.isin(["Climatology","Persistence","SeasonalNaive","AR12"])]
seed_sd = ml.groupby(["cell_id","fold","model"]).nse.std().dropna()
betw_sd = ml.groupby(["cell_id","fold"]).apply(lambda z: z.groupby("model").nse.mean().std()).dropna()
f,a = plt.subplots(figsize=(4.6,3.9))
a.hist(seed_sd,bins=40,alpha=.65,label=f"within-model, across seeds (med {seed_sd.median():.3f})")
a.hist(betw_sd,bins=40,alpha=.65,label=f"between-model (med {betw_sd.median():.3f})")
a.set_xlabel("s.d. of NSE"); a.set_ylabel("count"); a.legend(fontsize=7)
a.set_title("Fig 4  Is model choice bigger than random seed?")
f.savefig(FIG/"fig4_seed_vs_model.png"); plt.close(f)

# Fig 5 — leaked vs honest driver gain
f,axes = plt.subplots(1,2,figsize=(9,4),sharey=True)
for ax,s_,ttl in [(axes[0],sd0,"Drivers incl. concurrent month\n(diagnostic)"),
                  (axes[1],sd1,"Drivers lagged 1-3 months\n(forecast-honest)")]:
    for c_,gg in s_.groupby("irrig_class",observed=True): ax.scatter(gg.acf1_des,gg.driver_gain,s=14,label=f"irrig: {c_}",alpha=.8)
    x,y = s_.acf1_des.astype(float), s_.driver_gain.astype(float); ok=x.notna()&y.notna()
    sl,ic,r_,p_,_ = stats.linregress(x[ok],y[ok])
    xx=np.linspace(x[ok].min(),x[ok].max(),50); ax.plot(xx,sl*xx+ic,"k-",lw=1.2,label=f"r={r_:.2f}, p={p_:.2g}")
    ax.axhline(0,color="grey",lw=.8); ax.set_xlabel("deseasonalized lag-1 autocorrelation")
    ax.set_title(ttl,fontsize=9); ax.legend(fontsize=7)
axes[0].set_ylabel("NSE gain from drivers")
f.suptitle("Fig 5  Driver gain collapses once the concurrent month is removed",fontsize=10)
f.savefig(FIG/"fig5_driver_gain_both.png"); plt.close(f)

# Fig 6 — exceedance, honest
f,a = plt.subplots(figsize=(5.4,4.2))
s_ = sd1.sort_values("above_ceiling")
a.barh(range(len(s_)), s_.above_ceiling,
       color=["#c44" if v<0 else "#48a" for v in s_.above_ceiling], height=1.0)
a.axvline(0,color="k",lw=1)
a.set_xlabel("best NSE with lagged drivers  -  lag-only ceiling")
a.set_ylabel(f"cells (n={len(s_)}, sorted)"); a.set_yticks([])
a.set_title(f"Fig 6  {int((sd1.above_ceiling>0).sum())}/{len(sd1)} cells exceed the ceiling",fontsize=9)
f.savefig(FIG/"fig6_exceedance.png"); plt.close(f)

for p in sorted(FIG.glob("fig*.png")): print(f"{p.name}: {p.stat().st_size//1024} KB")

fig1_global_panel.png: 102 KB
fig2_ceiling_vs_achieved.png: 73 KB
fig3_ladder.png: 90 KB
fig4_seed_vs_model.png: 33 KB
fig5_driver_gain_both.png: 113 KB
fig6_exceedance.png: 24 KB


## Cell 10 — headline numbers (paste this output back)

In [ ]:
print("="*66); print("GLOBAL HEADLINE NUMBERS"); print("="*66)
print("\npanel N = %d cells | %d models | %d strata" % (res.cell_id.nunique(), res.model.nunique(), pan.stratum.nunique()))
print("lat %.1f to %.1f | AI %.3f-%.3f | P %.0f-%.0f mm/yr"
      % (pan.lat.min(),pan.lat.max(),pan.AI.min(),pan.AI.max(),pan.P_ann.min(),pan.P_ann.max()))
print("corr(AI, irrig_index) = %.3f" % stats.pearsonr(pan.AI,pan.irrig_index)[0])

print("\n[1] CEILING")
print("  panel mean %.3f | range %.3f-%.3f" % (ceil.ceil_lin_raw.mean(),ceil.ceil_lin_raw.min(),ceil.ceil_lin_raw.max()))
print("  corr(ceil_des, acf1_des) = %.3f" % ceil[["ceil_lin_des","acf1_des"]].corr().iloc[0,1])
print(ceil[ceil.replication][["name","ceil_lin_raw","ceil_lin_des","seas_share"]].round(3).to_string(index=False))

print("\n[2] BENCHMARK LADDER")
print(res.groupby("model").nse.agg(["mean","std"]).round(3).sort_values("mean",ascending=False).to_string())
print("\nwins by cell:\n", top.model.value_counts().to_string())

print("\n[3] SEED vs MODEL")
print("  seed sd %.4f | between-model sd %.4f | ratio %.1fx" % (seed_sd.median(),betw_sd.median(),betw_sd.median()/seed_sd.median()))
print("  corr(best NSE, ceiling) = %.3f" % rr_)

print("\n[4] STAGE D")
for tag,s_ in [("lag 1-3 (FORECAST)",sd1),("lag 0-2 (DIAGNOSTIC)",sd0)]:
    r_,p_ = stats.pearsonr(s_.acf1_des.astype(float), s_.driver_gain.astype(float))
    print("  %-22s %.3f -> %.3f | gain %+.3f | above ceiling %d/%d | corr(acf1,gain) %+.3f (p=%.2g)"
          % (tag, s_["lag_only"].mean(), s_["lag+drivers"].mean(), s_.driver_gain.mean(),
             int((s_.above_ceiling>0).sum()), len(s_), r_, p_))
print("\n  Punjab, forecast-honest:")
print(sd1[sd1.replication][["name","lag_only","lag+drivers","driver_gain","ceil_lin_raw","above_ceiling"]].round(3).to_string(index=False))

print("\n[5] EXCEEDANCE")
print("  %d/%d cells above ceiling | corr(seas_share, exceedance) = %.3f"
      % (int(top.over.sum()), len(top), stats.pearsonr(top.seas_share.astype(float), top.exceed.astype(float))[0]))

GLOBAL HEADLINE NUMBERS

panel N = 249 cells | 10 models | 29 strata
lat -55.2 to 64.0 | AI 0.001-2.828 | P 1-4021 mm/yr
corr(AI, irrig_index) = -0.073

[1] CEILING
  panel mean 0.751 | range 0.272-0.951
  corr(ceil_des, acf1_des) = 0.984
      name  ceil_lin_raw  ceil_lin_des  seas_share
  Firozpur         0.627         0.474       0.443
      Moga         0.664         0.510       0.476
Hoshiarpur         0.734         0.571       0.531

[2] BENCHMARK LADDER
                mean    std
model                      
AR12           0.687  0.210
RF             0.662  0.227
AdaBoost-CART  0.650  0.244
ESN            0.634  0.267
XGBoost        0.633  0.269
LightGBM       0.616  0.293
Persistence    0.490  0.213
Climatology    0.354  0.787
CART           0.348  0.500
SeasonalNaive  0.125  0.596

wins by cell:
 model
AR12             160
RF                51
AdaBoost-CART     16
ESN                9
XGBoost            6
LightGBM           4
Climatology        2
Persistence        1

[3] SEED

In [ ]:

import shutil
from google.colab import files
shutil.make_archive("figs", "zip", "figs")
files.download("figs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>